# DL Project Data Collection


### Author: Bradley Scott

This notebook builds the data for the prepared SPY datasets for training,
validation, and evaluation of return-quantile forecasting models.

Normalization parameters were fitted using training samples only.

#References:

Alpaca. (n.d.-a). [Historical bars](https://docs.alpaca.markets/us/reference/stockbars).

Alpaca. (n.d.-b). [Market data FAQ](https://docs.alpaca.markets/us/v1.1/docs/market-data-faq).

Alpaca. (n.d.). *[How to fetch historical market data with Alpaca’s Trading API](https://alpaca.markets/learn/fetch-historical-data).

Andersen, T. G., Bollerslev, T., Diebold, F. X., & Labys, P. (2003). [Modeling and forecasting realized volatility](https://doi.org/10.1111/1468-0262.00418). *Econometrica, 71*(2), 579–625.

pandas. (n.d.). [Windowing operations](https://pandas.pydata.org/docs/user_guide/window.html).

pandas_market_calendars. (n.d.). [Calendars](https://pandas-market-calendars.readthedocs.io/en/latest/usage.html).

scikit-learn. (n.d.-a). [Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html).

scikit-learn. (n.d.-b). [Time-related feature engineering](https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html).

scikit-learn. (n.d.-a). [Lagged features for time series forecasting](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).

Attri, P., Sharma, Y., Takach, K., & Shah, F. (2023). *[Timeseries forecasting for weather prediction](https://keras.io/examples/timeseries/timeseries_weather_forecasting/).* Keras.





##mount google drive and run requirements.txt file

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/SPY_DL_Project")
REQUIREMENTS = PROJECT_DIR / "requirements.txt"

assert REQUIREMENTS.is_file(), f"File not found: {REQUIREMENTS}"

%pip install -r {REQUIREMENTS}

##Import necessary modules

In [3]:
import alpaca
import pandas as pd
import numpy as np
import pyarrow as pa
import pandas_market_calendars as mcal
import time

from datetime import date, datetime, timedelta, timezone
from getpass import getpass
from tqdm.auto import tqdm

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.requests import CorporateActionsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import DataFeed


from google.colab import userdata

##connect to alpaca to get the data

In [4]:
# prompt for Alpaca credentials
api_key = getpass("Enter your Alpaca API Key: ")
secret_key = getpass("Enter your Alpaca Secret Key: ")

# initialize Alpaca client
client = StockHistoricalDataClient(
    api_key=api_key,
    secret_key=secret_key
)

print("Alpaca client initialized.")


Enter your Alpaca API Key: ··········
Enter your Alpaca Secret Key: ··········
Alpaca client initialized.


###download the necessary data

In [5]:
# NB: ALPACA doesn't have data in 2015
START_DATE = datetime(2016, 1, 1, tzinfo=timezone.utc)

# retrieve through August 31, 2026.
END_DATE = datetime(2026, 9, 1, tzinfo=timezone.utc)

# first date eligible for training samples.
SAMPLE_START_DATE = date(2016, 1, 1)

BATCH_DAYS = 7
all_batches = []

# download historical SPY data
batch_starts = []

current = START_DATE

while current < END_DATE:
    batch_starts.append(current)
    current += timedelta(days=BATCH_DAYS)

for batch_start in tqdm(batch_starts, desc="Downloading SPY"):

    batch_end = min(
        batch_start + timedelta(days=BATCH_DAYS),
        END_DATE
    )

    request = StockBarsRequest(
        symbol_or_symbols=["SPY"],
        timeframe=TimeFrame.Minute,
        start=batch_start,
        end=batch_end,
        feed=DataFeed.SIP,
        limit=10000,
        adjustment="raw"
    )

    # retry temporary API failures
    for attempt in range(3):
        try:
            bars = client.get_stock_bars(request)
            batch_df = bars.df

            if not batch_df.empty:
                all_batches.append(batch_df.reset_index())

            break

        except Exception as e:
            if attempt == 2:
                raise

            print(f"Retrying batch {batch_start.date()}: {e}")
            time.sleep(2 ** attempt)

    time.sleep(0.2)

# combine batches
if not all_batches:
    raise RuntimeError("No SPY data was downloaded.")

df = pd.concat(all_batches, ignore_index=True)

# convert timestamps to Eastern Time
df["timestamp"] = (
    pd.to_datetime(df["timestamp"], utc=True)
    .dt.tz_convert("America/New_York")
)

# sort chronologically and remove duplicates
df = (
    df.sort_values("timestamp")
      .drop_duplicates(subset=["symbol", "timestamp"])
      .reset_index(drop=True)
)

# keep regular trading hours
df = df.set_index("timestamp")

df = df.between_time(
    "09:30",
    "15:59"
)

df = df.reset_index()

# create trading date
df["trading_date"] = df["timestamp"].dt.date

# summarize
print("\nDownload complete!")
print("-" * 45)

print("Total minute bars:", f"{len(df):,}")
print("Trading days:", df["trading_date"].nunique())
print("First timestamp:", df["timestamp"].min())
print("Last timestamp:", df["timestamp"].max())

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())



Download complete!
---------------------------------------------
Total minute bars: 1,044,088
Trading days: 2680
First timestamp: 2016-01-04 09:30:00-05:00
Last timestamp: 2026-08-31 15:59:00-04:00

Columns:
['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'trading_date']

First five rows:


,timestamp,symbol,open,high,low,close,volume,trade_count,vwap,trading_date
0,2016-01-04 09:30:00-05:00,SPY,200.490,200.530,200.260,200.4100,2498430.0,4672.0,200.460265,2016-01-04
1,2016-01-04 09:31:00-05:00,SPY,200.405,200.436,200.205,200.3151,850216.0,3744.0,200.281888,2016-01-04
2,2016-01-04 09:32:00-05:00,SPY,200.310,200.460,200.310,200.4000,642808.0,2430.0,200.390525,2016-01-04
3,2016-01-04 09:33:00-05:00,SPY,200.400,200.410,200.200,200.2700,812281.0,3288.0,200.284406,2016-01-04
4,2016-01-04 09:34:00-05:00,SPY,200.270,200.270,200.020,200.0800,835983.0,3123.0,200.109604,2016-01-04



Last five rows:


,timestamp,symbol,open,high,low,close,volume,trade_count,vwap,trading_date
1044083,2026-08-31 15:55:00-04:00,SPY,767.360,767.360,767.00,767.120,801487.0,9353.0,767.125845,2026-08-31
1044084,2026-08-31 15:56:00-04:00,SPY,767.110,767.310,767.11,767.305,579314.0,7136.0,767.214767,2026-08-31
1044085,2026-08-31 15:57:00-04:00,SPY,767.310,767.420,767.16,767.180,610537.0,6616.0,767.342033,2026-08-31
1044086,2026-08-31 15:58:00-04:00,SPY,767.175,767.410,767.10,767.285,1095159.0,9985.0,767.273607,2026-08-31
1044087,2026-08-31 15:59:00-04:00,SPY,767.265,767.995,766.58,766.950,2388143.0,18565.0,767.512893,2026-08-31


###Basic validation of data

In [6]:
# count observations per trading day
daily_counts = (
    df.groupby("trading_date")
      .size()
      .rename("minute_bars")
)

print("Daily bar counts:")
display(daily_counts.describe())

print("\nSessions with fewer than 390 bars:")
display(daily_counts[daily_counts < 390])

print("\nMissing values:")
display(df.isna().sum())


Daily bar counts:


,minute_bars
count,2680.000000
mean,389.585075
std,4.828395
min,278.000000
25%,390.000000
50%,390.000000
75%,390.000000
max,390.000000



Sessions with fewer than 390 bars:


,minute_bars
trading_date,
2016-02-02,381
2016-11-25,342
2017-07-03,338
2017-11-24,337
2018-07-03,345
2018-11-23,335
2018-12-24,380
2019-07-03,344
2019-08-12,360



Missing values:


,0
timestamp,0
symbol,0
open,0
high,0
low,0
close,0
volume,0
trade_count,0
vwap,0
trading_date,0


###compare against NYSE calendar

In [7]:
# get the NYSE trading calendar

nyse = mcal.get_calendar("NYSE")

schedule = nyse.schedule(
    start_date=START_DATE.date(),
    end_date=(END_DATE - timedelta(days=1)).date()
)

# convert market open/close to Eastern Time
schedule["market_open"] = (
    schedule["market_open"].dt.tz_convert("America/New_York")
)

schedule["market_close"] = (
    schedule["market_close"].dt.tz_convert("America/New_York")
)

# prepare actual SPY observations
spy = df.copy()

spy["timestamp"] = pd.to_datetime(
    spy["timestamp"], utc=True
).dt.tz_convert("America/New_York")

spy["trading_date"] = spy["timestamp"].dt.date

# compare each day against scheduled hours
# group timestamps once, instead of repeatedly scanning the full dataset.
actual_by_date = {
    day: pd.DatetimeIndex(group["timestamp"]).sort_values()
    for day, group in spy.groupby("trading_date", sort=False)
}

empty_times = pd.DatetimeIndex(
    [],
    tz="America/New_York"
)

results = []

for trading_day, session in tqdm(
    schedule.iterrows(),
    total=len(schedule),
    desc="Validating NYSE sessions"
):
    day = trading_day.date()

    market_open = session["market_open"]
    market_close = session["market_close"]

    expected_times = pd.date_range(
        start=market_open,
        end=market_close,
        freq="min",
        inclusive="left"
    )

    actual_times = actual_by_date.get(day, empty_times)

    missing_times = expected_times.difference(actual_times)
    outside_times = actual_times.difference(expected_times)

    results.append({
        "date": day,
        "expected_bars": len(expected_times),
        "actual_bars": len(actual_times),
        "missing_bars": len(missing_times),
        "outside_session_bars": len(outside_times),
        "early_close": market_close.hour < 16,
        "first_missing": (
            missing_times.min() if len(missing_times) else pd.NaT
        ),
        "last_missing": (
            missing_times.max() if len(missing_times) else pd.NaT
        )
    })

validation = pd.DataFrame(results)

# print summary
print("NYSE Calendar Validation")
print("-" * 50)

print("Expected trading sessions:", len(schedule))
print("Downloaded trading sessions:", spy["trading_date"].nunique())

print("Sessions with missing bars:",
      (validation["missing_bars"] > 0).sum())

print("Total missing bars:",
      validation["missing_bars"].sum())

print("Sessions with outside-session bars:",
      (validation["outside_session_bars"] > 0).sum())

print("\nSessions requiring inspection:")

display(
    validation[
        (validation["missing_bars"] > 0) |
        (validation["outside_session_bars"] > 0)
    ]
)

# check unexpected or missing trading dates
expected_dates = set(schedule.index.date)
actual_dates = set(spy["trading_date"])

print("\nMissing trading dates:",
      sorted(expected_dates - actual_dates))

print("Unexpected trading dates:",
      sorted(actual_dates - expected_dates))


Validating NYSE sessions:   0%|          | 0/2680 [00:00<?, ?it/s]

NYSE Calendar Validation
--------------------------------------------------
Expected trading sessions: 2680
Downloaded trading sessions: 2680
Sessions with missing bars: 8
Total missing bars: 104
Sessions with outside-session bars: 21

Sessions requiring inspection:


,date,expected_bars,actual_bars,missing_bars,outside_session_bars,early_close,first_missing,last_missing
20,2016-02-02,390,381,9,0,False,2016-02-02 11:19:00-05:00,2016-02-02 11:27:00-05:00
227,2016-11-25,210,342,0,132,True,NaT,NaT
377,2017-07-03,210,338,0,128,True,NaT,NaT
478,2017-11-24,210,337,0,127,True,NaT,NaT
629,2018-07-03,210,345,0,135,True,NaT,NaT
729,2018-11-23,210,335,0,125,True,NaT,NaT
749,2018-12-24,210,380,0,170,True,NaT,NaT
880,2019-07-03,210,344,0,134,True,NaT,NaT
907,2019-08-12,390,360,30,0,False,2019-08-12 15:29:00-04:00,2019-08-12 15:59:00-04:00
984,2019-11-29,210,320,0,110,True,NaT,NaT



Missing trading dates: []
Unexpected trading dates: []


###inspect early close sessions

In [8]:
early_close_dates = validation.loc[
    validation["early_close"], "date"
]

early_close_data = (
    spy[spy["trading_date"].isin(early_close_dates)]
    .groupby("trading_date")
    .agg(
        first_bar=("timestamp", "min"),
        last_bar=("timestamp", "max"),
        total_bars=("timestamp", "count")
    )
)

display(early_close_data)

,first_bar,last_bar,total_bars
trading_date,,,
2016-11-25,2016-11-25 09:30:00-05:00,2016-11-25 15:59:00-05:00,342
2017-07-03,2017-07-03 09:30:00-04:00,2017-07-03 15:59:00-04:00,338
2017-11-24,2017-11-24 09:30:00-05:00,2017-11-24 15:58:00-05:00,337
2018-07-03,2018-07-03 09:30:00-04:00,2018-07-03 15:59:00-04:00,345
2018-11-23,2018-11-23 09:30:00-05:00,2018-11-23 15:59:00-05:00,335
2018-12-24,2018-12-24 09:30:00-05:00,2018-12-24 15:59:00-05:00,380
2019-07-03,2019-07-03 09:30:00-04:00,2019-07-03 15:59:00-04:00,344
2019-11-29,2019-11-29 09:30:00-05:00,2019-11-29 15:59:00-05:00,320
2019-12-24,2019-12-24 09:30:00-05:00,2019-12-24 15:59:00-05:00,278


In [9]:
missing_records = []

for trading_day, session in schedule.iterrows():

    day = trading_day.date()

    expected = pd.date_range(
        start=session["market_open"],
        end=session["market_close"],
        freq="min",
        inclusive="left"
    )

    actual = actual_by_date.get(day, empty_times)

    for missing_timestamp in expected.difference(actual):
        missing_records.append({
            "trading_date": day,
            "missing_timestamp": missing_timestamp
        })

missing_df = pd.DataFrame(
    missing_records,
    columns=["trading_date", "missing_timestamp"]
)

print("Total missing individual minutes:", len(missing_df))

display(missing_df.head(30))


Total missing individual minutes: 104


,trading_date,missing_timestamp
0,2016-02-02,2016-02-02 11:19:00-05:00
1,2016-02-02,2016-02-02 11:20:00-05:00
2,2016-02-02,2016-02-02 11:21:00-05:00
3,2016-02-02,2016-02-02 11:22:00-05:00
4,2016-02-02,2016-02-02 11:23:00-05:00
5,2016-02-02,2016-02-02 11:24:00-05:00
6,2016-02-02,2016-02-02 11:25:00-05:00
7,2016-02-02,2016-02-02 11:26:00-05:00
8,2016-02-02,2016-02-02 11:27:00-05:00
9,2019-08-12,2019-08-12 15:29:00-04:00


###Closer look at the outside-session bars

In [10]:
# identify early-close sessions
early_close_dates = validation.loc[
    validation["early_close"], "date"
].tolist()

# get original data for these sessions
early_df = df[
    df["trading_date"].isin(early_close_dates)
].copy()

# identify bars at or after 1:00 PM
outside_bars = early_df[
    early_df["timestamp"].dt.time >=
    pd.Timestamp("13:00").time()
].copy()

print("Total outside-session bars:", len(outside_bars))

# summarize each early-close session
summary = outside_bars.groupby("trading_date").agg(
    first_bar=("timestamp", "min"),
    last_bar=("timestamp", "max"),
    bar_count=("timestamp", "count"),
    total_volume=("volume", "sum"),
    median_volume=("volume", "median"),
    min_price=("low", "min"),
    max_price=("high", "max"),
    first_price=("open", "first"),
    last_price=("close", "last")
)

summary["price_change_pct"] = (
    (summary["last_price"] / summary["first_price"] - 1)
    * 100
)

display(summary)

Total outside-session bars: 2772


,first_bar,last_bar,bar_count,total_volume,median_volume,min_price,max_price,first_price,last_price,price_change_pct
trading_date,,,,,,,,,,
2016-11-25,2016-11-25 13:00:00-05:00,2016-11-25 15:59:00-05:00,132,9991805.0,4253.5,221.3261,221.8280,221.540,221.67,0.058680
2017-07-03,2017-07-03 13:00:00-04:00,2017-07-03 15:59:00-04:00,128,8335862.0,5223.5,241.7500,243.2900,242.310,241.83,-0.198093
2017-11-24,2017-11-24 13:00:00-05:00,2017-11-24 15:58:00-05:00,127,7952268.0,2700.0,259.8000,260.4500,260.350,260.12,-0.088343
2018-07-03,2018-07-03 13:00:00-04:00,2018-07-03 15:59:00-04:00,135,9341630.0,2600.0,270.2000,271.1300,270.820,270.30,-0.192009
2018-11-23,2018-11-23 13:00:00-05:00,2018-11-23 15:59:00-05:00,125,6661898.0,5031.0,262.7600,263.9757,263.370,262.94,-0.163268
2018-12-24,2018-12-24 13:00:00-05:00,2018-12-24 15:59:00-05:00,170,21003049.0,4523.5,232.6100,240.8355,234.410,233.07,-0.571648
2019-07-03,2019-07-03 13:00:00-04:00,2019-07-03 15:59:00-04:00,134,11741009.0,1988.5,298.4800,298.9500,298.750,298.72,-0.010042
2019-11-29,2019-11-29 13:00:00-05:00,2019-11-29 15:59:00-05:00,110,12598090.0,3429.0,314.1870,314.7600,314.270,314.50,0.073185
2019-12-24,2019-12-24 13:00:00-05:00,2019-12-24 15:59:00-05:00,68,4494707.0,3347.0,321.1600,321.3400,321.240,321.16,-0.024903


In [11]:
early_df["session_period"] = "Before 1 PM"

early_df.loc[
    early_df["timestamp"].dt.hour >= 13,
    "session_period"
] = "After 1 PM"

volume_comparison = (
    early_df.groupby(
        ["trading_date", "session_period"]
    )
    .agg(
        bars=("volume", "count"),
        total_volume=("volume", "sum"),
        average_volume=("volume", "mean"),
        median_volume=("volume", "median"),
        total_trades=("trade_count", "sum")
    )
)

display(volume_comparison)

bars  total_volume  average_volume  \
trading_date session_period                                       
2016-11-25   After 1 PM       132     9991805.0    75695.492424   
             Before 1 PM      210    29152050.0   138819.285714   
2017-07-03   After 1 PM       128     8335862.0    65123.921875   
             Before 1 PM      210    31543779.0   150208.471429   
2017-11-24   After 1 PM       127     7952268.0    62616.283465   
             Before 1 PM      210    19634597.0    93498.080952   
2018-07-03   After 1 PM       135     9341630.0    69197.259259   
             Before 1 PM      210    33697326.0   160463.457143   
2018-11-23   After 1 PM       125     6661898.0    53295.184000   
             Before 1 PM      210    35964613.0   171260.061905   
2018-12-24   After 1 PM       170    21003049.0   123547.347059   
             Before 1 PM      210   122940408.0   585430.514286   
2019-07-03   After 1 PM       134    11741009.0    87619.470149   
             Before 1 PM      210    29870777.0   142241.795238   
2019-11-29   After 1 PM       110    12598090.0   114528.090909   
             Before 1 PM      210    24817176.0   118177.028571   
2019-12-24   After 1 PM        68     4494707.0    66098.632353   
             Before 1 PM      210    15916112.0    75791.009524   
2020-11-27   After 1 PM       142     4757457.0    33503.218310   
             Before 1 PM      210    23356579.0   111221.804762   
2020-12-24   After 1 PM       113     6129642.0    54244.619469   
             Before 1 PM      210    20304785.0    96689.452381   
2021-11-26   After 1 PM       173    16018151.0    92590.468208   
             Before 1 PM      210    94207358.0   448606.466667   
2022-11-25   After 1 PM       120     5869626.0    48913.550000   
             Before 1 PM      210    23655084.0   112643.257143   
2023-07-03   After 1 PM        98     2049516.0    20913.428571   
             Before 1 PM      210    29789508.0   141854.800000   
2023-11-24   After 1 PM       119     4880654.0    41013.899160   
             Before 1 PM      210    24519735.0   116760.642857   
2024-07-03   After 1 PM       138     6396277.0    46349.833333   
             Before 1 PM      210    25828148.0   122991.180952   
2024-11-29   After 1 PM       134     8688696.0    64841.014925   
             Before 1 PM      210    20793986.0    99018.980952   
2024-12-24   After 1 PM       122    10202262.0    83625.098361   
             Before 1 PM      210    22291241.0   106148.766667   
2025-07-03   After 1 PM       167     5748104.0    34419.784431   
             Before 1 PM      210    43417898.0   206751.895238   
2025-11-28   After 1 PM       169    11308160.0    66912.189349   
             Before 1 PM      210    36875719.0   175598.661905   
2025-12-24   After 1 PM       148     4634901.0    31316.898649   
             Before 1 PM      210    34294147.0   163305.461905   

                             median_volume  total_trades  
trading_date session_period                               
2016-11-25   After 1 PM             4253.5        5103.0  
             Before 1 PM           89762.0      106917.0  
2017-07-03   After 1 PM             5223.5        5035.0  
             Before 1 PM           96368.0      127099.0  
2017-11-24   After 1 PM             2700.0        4585.0  
             Before 1 PM           54297.0       77910.0  
2018-07-03   After 1 PM             2600.0        5330.0  
             Before 1 PM          103456.5      173081.0  
2018-11-23   After 1 PM             5031.0        5028.0  
             Before 1 PM          125172.5      181792.0  
2018-12-24   After 1 PM             4523.5       15527.0  
             Before 1 PM          492807.5      657194.0  
2019-07-03   After 1 PM             1988.5        6122.0  
             Before 1 PM          101729.5      154206.0  
2019-11-29   After 1 PM             3429.0        6902.0  
             Before 1 PM           66638.5      105811.0  
2019-12-24   After

###Create a cleaned dataset that will only contain observations within the official NYSE trading hours.

In [12]:
# preserve original data
df_clean = df.copy()

# convert timestamps to UTC for comparison
df_clean["timestamp"] = pd.to_datetime(
    df_clean["timestamp"], utc=True
)

# prepare NYSE calendar schedule
session_schedule = schedule[
    ["market_open", "market_close"]
].copy()

session_schedule["trading_date"] = (
    session_schedule.index.date
)

# merge scheduled trading hours
df_clean = df_clean.merge(
    session_schedule[
        ["trading_date", "market_open", "market_close"]
    ],
    on="trading_date",
    how="left",
    validate="many_to_one"
)

# retain only observations inside scheduled trading hours
valid_session = (
    (df_clean["timestamp"] >= df_clean["market_open"]) &
    (df_clean["timestamp"] < df_clean["market_close"])
)

removed_bars = (~valid_session).sum()

df_clean = df_clean.loc[valid_session].copy()

# remove temporary calendar columns
df_clean = df_clean.drop(
    columns=["market_open", "market_close"]
)

# restore Eastern Time
df_clean["timestamp"] = (
    df_clean["timestamp"].dt.tz_convert(
        "America/New_York"
    )
)

df_clean = df_clean.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Removed outside-session bars:", removed_bars)
print("Cleaned total bars:", len(df_clean))
print("Trading sessions:", df_clean["trading_date"].nunique())

print("\nBars per session:")
print(
    df_clean.groupby("trading_date").size()
    .value_counts()
    .sort_index()
)

# verify uniqueness.
assert not df_clean.duplicated(
    ["symbol", "timestamp"]
).any(), "Duplicate symbol/timestamps detected."

# require every scheduled trading date to have some data.
expected_dates = set(schedule.index.date)
actual_dates = set(df_clean["trading_date"])

assert actual_dates == expected_dates, (
    f"Missing sessions: {sorted(expected_dates - actual_dates)}; "
    f"Unexpected sessions: {sorted(actual_dates - expected_dates)}"
)

# account for absent scheduled minutes without inventing bars.
expected_total = int(validation["expected_bars"].sum())
missing_total = int(validation["missing_bars"].sum())

assert len(df_clean) + missing_total == expected_total, (
    "Cleaned bars plus documented missing minutes "
    "do not match the calendar."
)

print("\nCalendar accounting checks passed.")
print("Expected scheduled minutes:", expected_total)
print("Observed regular-session bars:", len(df_clean))
print("Absent scheduled minutes:", missing_total)

print("\nSessions with absent scheduled minutes:")
display(
    validation.loc[
        validation["missing_bars"] > 0,
        [
            "date",
            "expected_bars",
            "missing_bars",
            "first_missing",
            "last_missing"
        ]
    ]
)

Removed outside-session bars: 2772
Cleaned total bars: 1041316
Trading sessions: 2680

Bars per session:
210      21
360       1
376       4
381       1
385       1
386       1
390    2651
Name: count, dtype: int64

Calendar accounting checks passed.
Expected scheduled minutes: 1041420
Observed regular-session bars: 1041316
Absent scheduled minutes: 104

Sessions with absent scheduled minutes:


,date,expected_bars,missing_bars,first_missing,last_missing
20,2016-02-02,390,9,2016-02-02 11:19:00-05:00,2016-02-02 11:27:00-05:00
907,2019-08-12,390,30,2019-08-12 15:29:00-04:00,2019-08-12 15:59:00-04:00
1051,2020-03-09,390,14,2020-03-09 09:35:00-04:00,2020-03-09 09:48:00-04:00
1054,2020-03-12,390,14,2020-03-12 09:36:00-04:00,2020-03-12 09:49:00-04:00
1056,2020-03-16,390,14,2020-03-16 09:31:00-04:00,2020-03-16 09:44:00-04:00
1058,2020-03-18,390,14,2020-03-18 12:57:00-04:00,2020-03-18 13:10:00-04:00
1343,2021-05-05,390,5,2021-05-05 11:27:00-04:00,2021-05-05 11:31:00-04:00
1867,2023-06-05,390,4,2023-06-05 09:52:00-04:00,2023-06-05 09:55:00-04:00


###price and volume checks

In [13]:
# work on a copy preserves df_clean
quality_df = df_clean.copy()

price_cols = ["open", "high", "low", "close", "vwap"]
numeric_cols = price_cols + ["volume", "trade_count"]

# check missing and non-finite values
missing_values = quality_df[numeric_cols].isna().sum()

nonfinite_values = pd.Series({
    col: (~np.isfinite(quality_df[col])).sum()
    for col in numeric_cols
})

# check invalid prices or activity
invalid_prices = (
    (quality_df[price_cols] <= 0).any(axis=1)
)

invalid_volume = quality_df["volume"] <= 0
invalid_trades = quality_df["trade_count"] <= 0

# check OHLC consistency
ohlc_errors = (
    (quality_df["high"] < quality_df["low"]) |
    (quality_df["high"] < quality_df["open"]) |
    (quality_df["high"] < quality_df["close"]) |
    (quality_df["low"] > quality_df["open"]) |
    (quality_df["low"] > quality_df["close"])
)

# VWAP should usually be within the minute's
# high-low range; flag violations for review.
vwap_outside_range = (
    (quality_df["vwap"] < quality_df["low"] - 0.0001) |
    (quality_df["vwap"] > quality_df["high"] + 0.0001)
)

# check duplicate timestamps
duplicate_bars = quality_df.duplicated(
    subset=["symbol", "timestamp"]
)

# summarize
integrity_summary = pd.DataFrame({
    "Check": [
        "Missing numeric values",
        "Non-finite numeric values",
        "Non-positive prices",
        "Non-positive volume",
        "Non-positive trade count",
        "OHLC inconsistencies",
        "VWAP outside high-low range",
        "Duplicate symbol/timestamps"
    ],
    "Flagged": [
        int(missing_values.sum()),
        int(nonfinite_values.sum()),
        int(invalid_prices.sum()),
        int(invalid_volume.sum()),
        int(invalid_trades.sum()),
        int(ohlc_errors.sum()),
        int(vwap_outside_range.sum()),
        int(duplicate_bars.sum())
    ]
})

display(integrity_summary)

print("\nRows with OHLC errors:")
display(quality_df.loc[ohlc_errors].head(20))

print("\nRows with VWAP range violations:")
display(quality_df.loc[vwap_outside_range].head(20))


,Check,Flagged
0,Missing numeric values,0
1,Non-finite numeric values,0
2,Non-positive prices,0
3,Non-positive volume,0
4,Non-positive trade count,0
5,OHLC inconsistencies,0
6,VWAP outside high-low range,0
7,Duplicate symbol/timestamps,0



Rows with OHLC errors:


,timestamp,symbol,open,high,low,close,volume,trade_count,vwap,trading_date



Rows with VWAP range violations:


,timestamp,symbol,open,high,low,close,volume,trade_count,vwap,trading_date


###check for unusual one-minute price movements

In [14]:
# sort by timestamp
quality_df = quality_df.sort_values(
    "timestamp"
).reset_index(drop=True)

# previous close within each trading session
quality_df["previous_close"] = (
    quality_df.groupby("trading_date")["close"]
    .shift(1)
)

# one-minute log return
quality_df["log_return"] = np.log(
    quality_df["close"] /
    quality_df["previous_close"]
)

quality_df["abs_return_pct"] = (
    quality_df["log_return"].abs() * 100
)

# identify unusual returns
large_moves = quality_df[
    quality_df["abs_return_pct"] > 0.5
].copy()

extreme_moves = quality_df[
    quality_df["abs_return_pct"] > 1.0
].copy()

print("Minute return statistics (%):")
display(
    (quality_df["log_return"] * 100).describe(
        percentiles=[0.01, 0.05, 0.95, 0.99, 0.999]
    )
)

print("\nMoves exceeding 0.5%:", len(large_moves))
print("Moves exceeding 1.0%:", len(extreme_moves))

print("\nTop 20 largest one-minute moves:")
display(
    quality_df.nlargest(
        20, "abs_return_pct"
    )[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "trade_count",
            "abs_return_pct"
        ]
    ]
)


Minute return statistics (%):


,log_return
count,1.038636e+06
mean,3.995123e-05
std,4.265047e-02
min,-1.489834e+00
1%,-1.209226e-01
5%,-5.950210e-02
50%,0.000000e+00
95%,5.774842e-02
99%,1.196473e-01
99.9%,2.717928e-01



Moves exceeding 0.5%: 261
Moves exceeding 1.0%: 25

Top 20 largest one-minute moves:


,timestamp,open,high,low,close,volume,trade_count,abs_return_pct
905585,2025-04-09 13:19:00-04:00,499.760,515.0500,498.6700,512.8000,5244918.0,57214.0,2.541782
406701,2020-03-03 10:00:00-05:00,305.940,312.8200,305.8200,312.7100,5869100.0,47845.0,2.188724
904616,2025-04-07 10:10:00-04:00,494.530,502.4200,494.3900,502.3837,1900894.0,18386.0,1.567547
904680,2025-04-07 11:14:00-04:00,503.780,504.4800,495.0700,496.3400,3594957.0,33757.0,1.489834
904624,2025-04-07 10:18:00-04:00,523.010,523.0900,515.5600,515.6400,2471924.0,23242.0,1.415350
413625,2020-03-26 15:50:00-04:00,257.160,261.4300,257.1600,260.7400,5779583.0,52838.0,1.390306
905586,2025-04-09 13:20:00-04:00,512.420,522.2900,507.3000,519.8500,3030132.0,29487.0,1.365440
410918,2020-03-17 15:59:00-04:00,250.900,254.4500,250.5401,254.1900,4512448.0,15402.0,1.298771
904619,2025-04-07 10:13:00-04:00,510.650,517.3300,509.9800,517.2200,2300484.0,21363.0,1.288181
905594,2025-04-09 13:28:00-04:00,531.790,531.7951,524.9400,525.1600,2117758.0,18702.0,1.254570


###check volume and trading activity

In [15]:
# average shares per reported trade
quality_df["shares_per_trade"] = (
    quality_df["volume"] /
    quality_df["trade_count"].replace(0, np.nan)
)

# volume percentiles
print("One-minute volume distribution:")
display(
    quality_df["volume"].describe(
        percentiles=[
            0.01, 0.05, 0.50,
            0.95, 0.99, 0.999
        ]
    )
)

# unusually high volume
volume_threshold = quality_df["volume"].quantile(0.999)

high_volume = quality_df[
    quality_df["volume"] > volume_threshold
]

print("\n99.9th percentile volume threshold:",
      f"{volume_threshold:,.0f}")

print("Extreme-volume bars:", len(high_volume))

print("\nTop 20 volume observations:")
display(
    quality_df.nlargest(20, "volume")[
        [
            "timestamp",
            "open",
            "close",
            "volume",
            "trade_count",
            "shares_per_trade"
        ]
    ]
)

print("\nShares-per-trade distribution:")
display(
    quality_df["shares_per_trade"].describe(
        percentiles=[0.01, 0.50, 0.99, 0.999]
    )
)

print("\nLowest-volume observations:")
display(
    quality_df.nsmallest(20, "volume")[
        [
            "timestamp",
            "close",
            "volume",
            "trade_count"
        ]
    ]
)


One-minute volume distribution:


,volume
count,1.041316e+06
mean,1.771579e+05
std,2.437521e+05
min,1.579000e+03
1%,1.798100e+04
5%,2.999300e+04
50%,1.108390e+05
95%,5.361760e+05
99%,1.089729e+06
99.9%,2.756413e+06



99.9th percentile volume threshold: 2,756,413
Extreme-volume bars: 1042

Top 20 volume observations:


,timestamp,open,close,volume,trade_count,shares_per_trade
388191,2019-12-20 10:30:00-05:00,320.7933,320.8600,23934399.0,632.0,37870.884494
266494,2018-09-21 10:43:00-04:00,292.7100,292.6900,18660503.0,573.0,32566.322862
94231,2016-12-16 10:10:00-05:00,225.6200,225.6299,16035388.0,1001.0,16019.368631
411740,2020-03-20 10:25:00-04:00,241.7200,241.9650,15918133.0,7756.0,2052.363718
240173,2018-06-15 11:02:00-04:00,276.2100,276.2100,15802536.0,677.0,23342.002954
778060,2023-12-15 10:54:00-05:00,470.2900,470.3800,12603971.0,2043.0,6169.344591
200343,2018-01-19 10:12:00-05:00,279.5700,279.5900,12337924.0,693.0,17803.642136
492493,2021-01-15 10:18:00-05:00,374.9500,374.5400,12321177.0,2447.0,5035.217409
191770,2017-12-15 10:19:00-05:00,266.2500,266.2400,11993600.0,1494.0,8027.844712
592164,2022-01-21 10:34:00-05:00,444.8800,445.0700,11803033.0,6276.0,1880.661727



Shares-per-trade distribution:


,shares_per_trade
count,1.041316e+06
mean,1.579531e+02
std,2.504945e+02
min,3.711348e+00
1%,3.782590e+01
50%,1.223757e+02
99%,8.143869e+02
99.9%,3.077921e+03
max,3.793933e+04



Lowest-volume observations:


,timestamp,close,volume,trade_count
173772,2017-10-10 12:51:00-04:00,254.3200,1579.0,22.0
156276,2017-08-07 13:45:00-04:00,247.6950,1642.0,21.0
175719,2017-10-17 12:48:00-04:00,255.2000,1688.0,21.0
130278,2017-05-02 12:57:00-04:00,238.8100,1962.0,31.0
96374,2016-12-23 13:23:00-05:00,225.3850,1981.0,22.0
176229,2017-10-18 14:48:00-04:00,255.8200,2011.0,33.0
176155,2017-10-18 13:34:00-04:00,255.8432,2088.0,23.0
149597,2017-07-13 12:56:00-04:00,244.4500,2089.0,28.0
174215,2017-10-11 13:44:00-04:00,254.6500,2117.0,28.0
173342,2017-10-09 12:11:00-04:00,254.3000,2297.0,30.0


###Check for abnormal price ranges and suspicious VWAP

In [16]:
# intraminute high-low range as a percentage of open
quality_df["range_pct"] = (
    (quality_df["high"] - quality_df["low"]) /
    quality_df["open"] * 100
)

# VWAP deviation from close
quality_df["vwap_deviation_pct"] = (
    (quality_df["vwap"] - quality_df["close"]).abs() /
    quality_df["close"] * 100
)

print("High-low range distribution (%):")
display(
    quality_df["range_pct"].describe(
        percentiles=[0.95, 0.99, 0.999]
    )
)

print("\nTop 20 widest minute ranges:")
display(
    quality_df.nlargest(20, "range_pct")[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "range_pct"
        ]
    ]
)

print("\nTop 20 VWAP deviations:")
display(
    quality_df.nlargest(20, "vwap_deviation_pct")[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "vwap",
            "vwap_deviation_pct"
        ]
    ]
)


High-low range distribution (%):


,range_pct
count,1.041316e+06
mean,4.971711e-02
std,4.850904e-02
min,0.000000e+00
50%,3.659071e-02
95%,1.288187e-01
99%,2.304275e-01
99.9%,4.823781e-01
max,3.277573e+00



Top 20 widest minute ranges:


,timestamp,open,high,low,close,volume,range_pct
905585,2025-04-09 13:19:00-04:00,499.760,515.050,498.6700,512.8000,5244918.0,3.277573
410454,2020-03-16 14:45:00-04:00,247.610,248.420,240.7800,247.6200,632976.0,3.085497
486230,2020-12-21 13:25:00-05:00,367.530,378.460,367.3300,367.4600,1813459.0,3.028324
905586,2025-04-09 13:20:00-04:00,512.420,522.290,507.3000,519.8500,3030132.0,2.925335
1023765,2026-06-26 15:59:00-04:00,733.400,733.405,716.5800,729.0900,4700697.0,2.294110
406701,2020-03-03 10:00:00-05:00,305.940,312.820,305.8200,312.7100,5869100.0,2.288030
410963,2020-03-18 10:14:00-04:00,243.340,248.370,243.1800,244.0310,413662.0,2.132818
411243,2020-03-18 15:08:00-04:00,234.550,235.680,230.7143,234.7800,1212763.0,2.117118
410954,2020-03-18 10:05:00-04:00,245.020,245.140,240.0000,244.3900,620622.0,2.097788
426515,2020-05-14 09:40:00-04:00,278.590,278.590,272.9900,278.3800,570903.0,2.010122



Top 20 VWAP deviations:


,timestamp,open,high,low,close,vwap,vwap_deviation_pct
905585,2025-04-09 13:19:00-04:00,499.760,515.0500,498.6700,512.8000,507.097046,1.112121
905586,2025-04-09 13:20:00-04:00,512.420,522.2900,507.3000,519.8500,514.155171,1.095475
69356,2016-09-15 15:05:00-04:00,215.670,215.6700,213.1551,215.6500,213.697504,0.905400
69352,2016-09-15 15:01:00-04:00,215.540,215.6400,213.1551,215.6350,213.747672,0.875242
406701,2020-03-03 10:00:00-05:00,305.940,312.8200,305.8200,312.7100,310.232035,0.792416
410532,2020-03-17 09:33:00-04:00,246.740,246.8300,244.0700,244.0800,245.952179,0.767035
904619,2025-04-07 10:13:00-04:00,510.650,517.3300,509.9800,517.2200,513.483843,0.722354
904616,2025-04-07 10:10:00-04:00,494.530,502.4200,494.3900,502.3837,499.001589,0.673213
905588,2025-04-09 13:22:00-04:00,519.500,521.4800,513.6300,521.4500,517.964226,0.668477
412733,2020-03-24 13:58:00-04:00,237.690,239.4910,237.3100,237.3178,238.845639,0.643795


###Final validation summary

In [17]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Total trading sessions",
        "Total minute bars",
        "Missing numeric values",
        "OHLC inconsistencies",
        "Duplicate timestamps",
        "Non-positive price rows",
        "Non-positive volume rows",
        "VWAP range violations",
        "Minute moves > 0.5%",
        "Minute moves > 1.0%",
        "Minute ranges > 1.0%"
    ],
    "Value": [
        quality_df["trading_date"].nunique(),
        len(quality_df),
        int(missing_values.sum()),
        int(ohlc_errors.sum()),
        int(duplicate_bars.sum()),
        int(invalid_prices.sum()),
        int(invalid_volume.sum()),
        int(vwap_outside_range.sum()),
        int((quality_df["abs_return_pct"] > 0.5).sum()),
        int((quality_df["abs_return_pct"] > 1.0).sum()),
        int((quality_df["range_pct"] > 1.0).sum())
    ]
})

display(quality_summary)

,Metric,Value
0,Total trading sessions,2680
1,Total minute bars,1041316
2,Missing numeric values,0
3,OHLC inconsistencies,0
4,Duplicate timestamps,0
5,Non-positive price rows,0
6,Non-positive volume rows,0
7,VWAP range violations,0
8,Minute moves > 0.5%,261
9,Minute moves > 1.0%,25


###investigate unusual high-low observations

In [18]:
# investigate suspicious high-low observations

suspicious_times = [
    "2025-05-27 13:13",
    "2026-06-26 15:59"
]

for ts in suspicious_times:
    target = pd.Timestamp(ts, tz="America/New_York")

    window = quality_df[
        (quality_df["timestamp"] >= target - pd.Timedelta(minutes=5)) &
        (quality_df["timestamp"] <= target + pd.Timedelta(minutes=5))
    ][[
        "timestamp", "open", "high", "low",
        "close", "vwap", "volume", "trade_count"
    ]]

    print(f"\nObservations around {target}")
    display(window)


Observations around 2025-05-27 13:13:00-04:00


,timestamp,open,high,low,close,vwap,volume,trade_count
918054,2025-05-27 13:08:00-04:00,590.180,590.23,590.1000,590.140,590.155970,71663.0,929.0
918055,2025-05-27 13:09:00-04:00,590.130,590.15,590.0600,590.120,590.097060,54098.0,743.0
918056,2025-05-27 13:10:00-04:00,590.110,590.17,589.9500,589.960,590.062037,62670.0,932.0
918057,2025-05-27 13:11:00-04:00,589.980,590.01,589.9000,589.925,589.944246,59911.0,886.0
918058,2025-05-27 13:12:00-04:00,589.920,589.98,589.9000,589.970,589.950310,98770.0,936.0
918059,2025-05-27 13:13:00-04:00,589.980,590.11,578.4300,590.100,589.735195,124287.0,1461.0
918060,2025-05-27 13:14:00-04:00,590.100,590.19,585.3864,590.050,589.993341,98629.0,1231.0
918061,2025-05-27 13:15:00-04:00,590.065,590.10,589.9900,590.050,590.051441,60701.0,903.0
918062,2025-05-27 13:16:00-04:00,590.050,590.15,589.9800,590.060,590.084431,131074.0,1311.0
918063,2025-05-27 13:17:00-04:00,590.070,590.12,590.0400,590.110,590.079100,59862.0,895.0



Observations around 2026-06-26 15:59:00-04:00


,timestamp,open,high,low,close,vwap,volume,trade_count
1023760,2026-06-26 15:54:00-04:00,733.120,733.240,732.78,733.150,733.024723,348678.0,5868.0
1023761,2026-06-26 15:55:00-04:00,733.130,733.360,732.92,733.090,733.124945,550735.0,7743.0
1023762,2026-06-26 15:56:00-04:00,733.095,733.150,732.94,733.145,733.085856,504151.0,4843.0
1023763,2026-06-26 15:57:00-04:00,733.145,733.570,733.13,733.340,733.345443,543125.0,6977.0
1023764,2026-06-26 15:58:00-04:00,733.330,733.600,733.08,733.410,733.368315,577108.0,8986.0
1023765,2026-06-26 15:59:00-04:00,733.400,733.405,716.58,729.090,730.071091,4700697.0,40499.0


###Flag the suspicious high low observations as anomalies but dont remove them from df_clean

In [19]:
# flag suspicious observations without altering prices

df_clean["suspected_price_anomaly"] = False

anomalies = [
    "2025-05-27 13:13",
    "2025-05-27 13:14",
    "2026-06-26 15:59"
]

anomaly_timestamps = pd.to_datetime(anomalies).tz_localize(
    "America/New_York"
)

df_clean.loc[
    df_clean["timestamp"].isin(anomaly_timestamps),
    "suspected_price_anomaly"
] = True

display(
    df_clean.loc[
        df_clean["suspected_price_anomaly"],
        [
            "timestamp", "open", "high",
            "low", "close", "volume",
            "suspected_price_anomaly"
        ]
    ]
)

,timestamp,open,high,low,close,volume,suspected_price_anomaly
918059,2025-05-27 13:13:00-04:00,589.98,590.110,578.4300,590.10,124287.0,True
918060,2025-05-27 13:14:00-04:00,590.10,590.190,585.3864,590.05,98629.0,True
1023765,2026-06-26 15:59:00-04:00,733.40,733.405,716.5800,729.09,4700697.0,True


##Pull SPY corporated action and dividend validation

In [20]:
# reuse credentials already entered with getpass()
ca_client = CorporateActionsClient(
    api_key=api_key,
    secret_key=secret_key
)

request = CorporateActionsRequest(
    symbols=["SPY"],
    start=START_DATE.date(),
    end=END_DATE.date(),
    limit=1000
)

actions = ca_client.get_corporate_actions(request)

# convert to DataFrame
corporate_actions = actions.df.copy()

print("Corporate actions retrieved:", len(corporate_actions))
print("\nColumns:")
print(corporate_actions.columns.tolist())

display(corporate_actions)

Corporate actions retrieved: 40

Columns:
['id', 'symbol', 'cusip', 'rate', 'special', 'foreign', 'process_date', 'ex_date', 'record_date', 'payable_date']


,id,symbol,cusip,rate,special,foreign,process_date,ex_date,record_date,payable_date
corporate_action_type,,,,,,,,,,
cash_dividends,c3b774ec-c4d6-4ef7-b262-c1ed32867b37,SPY,,1.078442,False,False,2016-06-17,2016-06-17,None,None
cash_dividends,74a2365b-6208-40b1-916a-986de2305df8,SPY,,1.082068,False,False,2016-09-16,2016-09-16,None,None
cash_dividends,623ffd40-1246-4262-8b22-f551b550697c,SPY,,1.328930,False,False,2016-12-16,2016-12-16,None,None
cash_dividends,80059a76-66b9-4d92-b282-2660524b4479,SPY,,1.033118,False,False,2017-03-17,2017-03-17,None,None
cash_dividends,74dd6dd6-d8b1-4bc3-8e74-668d5b68600f,SPY,,1.183108,False,False,2017-06-16,2017-06-16,None,None
cash_dividends,6e241cab-e311-4a61-bb60-9b26ab595a7b,SPY,,1.234574,False,False,2017-09-15,2017-09-15,None,None
cash_dividends,b4ef9c15-5209-4829-8c1c-74f73aec0cfa,SPY,,1.351333,False,False,2017-12-15,2017-12-15,None,None
cash_dividends,4336bcd9-00b9-46e6-bba9-04cd20c44936,SPY,,1.096775,False,False,2018-03-16,2018-03-16,None,None
cash_dividends,8fb7aaec-2195-4176-a89b-7942b7e69602,SPY,,1.322611,False,False,2018-09-21,2018-09-21,None,None


###inspect dividend dates and overnight price changes

In [21]:
# work on our already validated, cleaned dataset
audit_df = df_clean.copy()

daily = (
    audit_df.sort_values("timestamp")
    .groupby("trading_date")
    .agg(
        first_open=("open", "first"),
        last_close=("close", "last"),
        daily_high=("high", "max"),
        daily_low=("low", "min"),
        daily_volume=("volume", "sum")
    )
)

# prior session closing price
daily["previous_close"] = daily["last_close"].shift(1)

# overnight opening gap percentage
daily["overnight_gap_pct"] = (
    daily["first_open"] / daily["previous_close"] - 1
) * 100

print("Overnight gap distribution:")
display(daily["overnight_gap_pct"].describe())

print("\nLargest negative overnight gaps:")
display(daily.nsmallest(20, "overnight_gap_pct"))

print("\nLargest positive overnight gaps:")
display(daily.nlargest(20, "overnight_gap_pct"))

Overnight gap distribution:


,overnight_gap_pct
count,2679.000000
mean,0.033274
std,0.727429
min,-10.980696
25%,-0.226530
50%,0.065991
75%,0.342881
max,6.041919



Largest negative overnight gaps:


,first_open,last_close,daily_high,daily_low,daily_volume,previous_close,overnight_gap_pct
trading_date,,,,,,,
2020-03-16,241.18,239.41,256.900,237.36,262416618.0,270.93,-10.980696
2020-03-09,275.29,274.40,284.190,273.45,276298662.0,297.42,-7.440656
2020-03-18,236.25,241.01,248.370,228.02,298244128.0,254.19,-7.057713
2020-03-12,256.00,248.10,266.660,247.68,345218710.0,274.36,-6.691938
2024-08-05,511.64,517.41,523.580,510.27,112728886.0,532.98,-4.003903
2020-04-01,247.98,246.12,251.440,243.90,169930843.0,257.70,-3.771828
2025-04-03,545.11,537.64,547.970,536.90,102508061.0,564.65,-3.460551
2016-06-24,203.62,203.22,206.905,202.72,287434839.0,210.83,-3.419817
2025-04-07,489.19,504.55,523.170,481.80,220789128.0,505.50,-3.226508



Largest positive overnight gaps:


,first_open,last_close,daily_high,daily_low,daily_volume,previous_close,overnight_gap_pct
trading_date,,,,,,,
2020-03-13,263.09,270.930,271.08,249.580,284728893.0,248.10,6.041919
2020-03-24,234.42,243.590,244.10,233.800,211683317.0,222.51,5.352568
2020-11-09,363.97,354.540,364.38,354.060,143508089.0,350.19,3.935007
2020-04-06,257.84,264.960,267.00,256.725,158092423.0,248.21,3.879779
2020-03-10,284.62,288.440,288.52,273.500,246106243.0,274.40,3.724490
2022-11-10,388.03,394.640,395.04,385.642,114056299.0,374.15,3.709742
2020-04-07,274.21,265.100,275.03,264.890,179135157.0,264.96,3.491093
2025-04-08,521.87,496.740,524.98,489.160,143796337.0,504.55,3.432762
2025-05-12,581.49,582.840,583.00,577.040,65396569.0,564.29,3.048078


###compare raw versus dividend-adjusted prices

In [22]:
def get_daily_spy(adjustment):
    request = StockBarsRequest(
        symbol_or_symbols=["SPY"],
        timeframe=TimeFrame.Day,
        start=datetime(2024, 1, 1, tzinfo=timezone.utc),
        end=datetime(2024, 7, 1, tzinfo=timezone.utc),
        feed=DataFeed.SIP,
        adjustment=adjustment
    )

    result = client.get_stock_bars(request)
    return result.df.reset_index()

raw_daily = get_daily_spy("raw")
adjusted_daily = get_daily_spy("all")

comparison = raw_daily[
    ["timestamp", "close"]
].merge(
    adjusted_daily[["timestamp", "close"]],
    on="timestamp",
    suffixes=("_raw", "_adjusted")
)

comparison["adjustment_ratio"] = (
    comparison["close_adjusted"] /
    comparison["close_raw"]
)

display(comparison.head(20))
display(comparison.tail(20))

,timestamp,close_raw,close_adjusted,adjustment_ratio
0,2024-01-02 05:00:00+00:00,472.65,457.65,0.968264
1,2024-01-03 05:00:00+00:00,468.79,453.91,0.968259
2,2024-01-04 05:00:00+00:00,467.28,452.45,0.968263
3,2024-01-05 05:00:00+00:00,467.92,453.07,0.968264
4,2024-01-08 05:00:00+00:00,474.60,459.54,0.968268
5,2024-01-09 05:00:00+00:00,473.88,458.84,0.968262
6,2024-01-10 05:00:00+00:00,476.56,461.44,0.968273
7,2024-01-11 05:00:00+00:00,476.35,461.23,0.968259
8,2024-01-12 05:00:00+00:00,476.68,461.55,0.968260
9,2024-01-16 05:00:00+00:00,474.93,459.86,0.968269


,timestamp,close_raw,close_adjusted,adjustment_ratio
104,2024-05-31 04:00:00+00:00,527.37,512.22,0.971273
105,2024-06-03 04:00:00+00:00,527.80,512.64,0.971277
106,2024-06-04 04:00:00+00:00,528.39,513.21,0.971271
107,2024-06-05 04:00:00+00:00,534.67,519.31,0.971272
108,2024-06-06 04:00:00+00:00,534.66,519.30,0.971271
109,2024-06-07 04:00:00+00:00,534.01,518.67,0.971274
110,2024-06-10 04:00:00+00:00,535.66,520.27,0.971269
111,2024-06-11 04:00:00+00:00,536.95,521.53,0.971282
112,2024-06-12 04:00:00+00:00,541.36,525.81,0.971276
113,2024-06-13 04:00:00+00:00,542.45,526.87,0.971278


###build out a new df that has the dividend information so we can calculate an approzimate dividend-neutral overnight return.

In [23]:
# get cash dividend records
dividends = (
    corporate_actions
    .reset_index()
    .query("corporate_action_type == 'cash_dividends'")
    [["ex_date", "rate"]]
    .copy()
)

dividends["ex_date"] = pd.to_datetime(
    dividends["ex_date"]
).dt.date

dividends["rate"] = pd.to_numeric(
    dividends["rate"]
)

# retain dividends within the downloaded period.
dividends = dividends[
    (dividends["ex_date"] >= START_DATE.date())
    & (dividends["ex_date"] < END_DATE.date())
].copy()

# multiple dividends on one date, if any
dividends = (
    dividends.groupby("ex_date", as_index=False)
    ["rate"].sum()
    .rename(columns={
        "ex_date": "trading_date",
        "rate": "dividend_per_share"
    })
)

# preserve original cleaned data
df_prepared = df_clean.copy()

# add dividend information
df_prepared = df_prepared.merge(
    dividends,
    on="trading_date",
    how="left",
    validate="many_to_one"
)

df_prepared["dividend_per_share"] = (
    df_prepared["dividend_per_share"].fillna(0.0)
)

df_prepared["is_ex_dividend"] = (
    df_prepared["dividend_per_share"] > 0
)

# calculate previous-session closing price
daily_reference = (
    df_prepared.sort_values("timestamp")
    .groupby("trading_date")
    .agg(
        session_open=("open", "first"),
        session_close=("close", "last"),
        dividend=("dividend_per_share", "first")
    )
)

daily_reference["previous_close"] = (
    daily_reference["session_close"].shift(1)
)

# raw overnight price gap
daily_reference["overnight_raw_pct"] = (
    daily_reference["session_open"] /
    daily_reference["previous_close"] - 1
) * 100

# approximate dividend-inclusive overnight return
daily_reference["overnight_div_neutral_pct"] = (
    (
        daily_reference["session_open"] +
        daily_reference["dividend"]
    ) / daily_reference["previous_close"] - 1
) * 100

print("Ex-dividend sessions:")
display(
    daily_reference[daily_reference["dividend"] > 0]
    [[
        "dividend",
        "previous_close",
        "session_open",
        "overnight_raw_pct",
        "overnight_div_neutral_pct"
    ]]
)

print("\nPrepared dataset:")
print("Rows:", len(df_prepared))
print("Trading sessions:", df_prepared["trading_date"].nunique())
print("Dividend sessions:", df_prepared.loc[
    df_prepared["is_ex_dividend"],
    "trading_date"
].nunique())

Ex-dividend sessions:


,dividend,previous_close,session_open,overnight_raw_pct,overnight_div_neutral_pct
trading_date,,,,,
2016-06-17,1.078442,208.420,207.17,-0.599751,-0.082314
2016-09-16,1.082068,215.290,213.48,-0.840726,-0.338117
2016-12-16,1.328930,226.770,226.00,-0.339551,0.246474
2017-03-17,1.033118,238.520,237.75,-0.322824,0.110313
2017-06-16,1.183108,243.790,242.77,-0.418393,0.066905
2017-09-15,1.234574,250.100,248.69,-0.563774,-0.070142
2017-12-15,1.351333,265.670,265.45,-0.082810,0.425841
2018-03-16,1.096775,275.010,274.48,-0.192720,0.206093
2018-09-21,1.322611,293.550,293.09,-0.156702,0.293855



Prepared dataset:
Rows: 1041316
Trading sessions: 2680
Dividend sessions: 40


##Feature Engineering

###Initialize feature dataset

In [24]:
# preserve the original prepared data
df_features = df_prepared.copy()

# ensure timestamps are in Eastern Time
df_features["timestamp"] = pd.to_datetime(
    df_features["timestamp"], utc=True
).dt.tz_convert("America/New_York")

# sort chronologically
df_features = (
    df_features.sort_values("timestamp")
    .reset_index(drop=True)
)

# define the 18 input features
FEATURE_COLUMNS = [
    "log_return_1m",
    "candle_range",
    "candle_body",
    "upper_wick",
    "lower_wick",
    "log_volume",
    "relative_volume_20d",
    "momentum_5m",
    "momentum_15m",
    "momentum_30m",
    "volatility_5m",
    "volatility_15m",
    "volatility_30m",
    "time_sin",
    "time_cos",
    "overnight_gap",
    "session_range_position",
    "vwap_deviation"
]

# create time-index columns
df_features["minute_of_day"] = (
    df_features["timestamp"].dt.hour * 60
    + df_features["timestamp"].dt.minute
)

df_features["minute_index"] = (
    df_features["minute_of_day"] - 570
)

# validate initialization
print("Rows:", len(df_features))
print(
    "Trading sessions:",
    df_features["trading_date"].nunique()
)
print("Feature count:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 18
assert not df_features.duplicated(
    ["symbol", "timestamp"]
).any()

print("\nTime index verification:")
display(
    df_features[
        [
            "timestamp",
            "trading_date",
            "minute_index",
            "minute_of_day"
        ]
    ].head(10)
)

Rows: 1041316
Trading sessions: 2680
Feature count: 18

Time index verification:


,timestamp,trading_date,minute_index,minute_of_day
0,2016-01-04 09:30:00-05:00,2016-01-04,0,570
1,2016-01-04 09:31:00-05:00,2016-01-04,1,571
2,2016-01-04 09:32:00-05:00,2016-01-04,2,572
3,2016-01-04 09:33:00-05:00,2016-01-04,3,573
4,2016-01-04 09:34:00-05:00,2016-01-04,4,574
5,2016-01-04 09:35:00-05:00,2016-01-04,5,575
6,2016-01-04 09:36:00-05:00,2016-01-04,6,576
7,2016-01-04 09:37:00-05:00,2016-01-04,7,577
8,2016-01-04 09:38:00-05:00,2016-01-04,8,578
9,2016-01-04 09:39:00-05:00,2016-01-04,9,579


###Price, candle and volume features

In [25]:
# one-minute log returns
sessions = df_features.groupby("trading_date", sort=False)

previous_close = sessions["close"].shift(1)
previous_time = sessions["timestamp"].shift(1)

# a one-minute return requires consecutive minute bars.
adjacent = (
    df_features["timestamp"] - previous_time
).eq(pd.Timedelta(minutes=1))

# the scheduled 09:30 bar uses its own opening price.
opening_bar = df_features["minute_index"].eq(0)

return_reference = previous_close.where(adjacent)
return_reference = return_reference.where(
    ~opening_bar,
    df_features["open"]
)

df_features["log_return_1m"] = np.log(
    df_features["close"] / return_reference
)

# candle properties
df_features["candle_range"] = (
    (df_features["high"] - df_features["low"])
    / df_features["open"]
)

df_features["candle_body"] = (
    (df_features["close"] - df_features["open"])
    / df_features["open"]
)

df_features["upper_wick"] = (
    df_features["high"]
    - df_features[["open", "close"]].max(axis=1)
) / df_features["open"]

df_features["lower_wick"] = (
    df_features[["open", "close"]].min(axis=1)
    - df_features["low"]
) / df_features["open"]


# log volume
df_features["log_volume"] = np.log1p(
    df_features["volume"]
)

# price relative to minute VWAP
df_features["vwap_deviation"] = (
    df_features["close"] / df_features["vwap"] - 1
)

print("Price, candle and volume features calculated.")

Price, candle and volume features calculated.


###relative volume using previous 20 sessions

In [26]:
# recalculate relative volume using the previous
# 20 available observations at each minute of day.
volume_matrix = df_features.pivot(
    index="trading_date",
    columns="minute_of_day",
    values="volume"
).sort_index()

# for each minute, calculate the mean of the previous
# 20 non-missing session observations.
historical_mean = volume_matrix.apply(
    lambda col: col.dropna()
        .shift(1)
        .rolling(window=20, min_periods=20)
        .mean()
        .reindex(volume_matrix.index)
)

relative_volume_matrix = (
    volume_matrix / historical_mean
)

# convert to long format without deprecated stack()
relative_volume_long = (
    relative_volume_matrix
    .rename_axis(index="trading_date", columns="minute_of_day")
    .melt(
        ignore_index=False,
        var_name="minute_of_day",
        value_name="relative_volume_20d"
    )
    .reset_index()
)

# replace old relative volume column
df_features = df_features.drop(
    columns=["relative_volume_20d"],
    errors="ignore"
)

df_features = df_features.merge(
    relative_volume_long,
    on=["trading_date", "minute_of_day"],
    how="left",
    validate="one_to_one"
)

print(
    "Missing relative volume values:",
    df_features["relative_volume_20d"].isna().sum()
)


Missing relative volume values: 7800


###multiscale momentum and realized volatility

NB: had to do rolling expanding window momentum and volatility features instead of 30 minute momentum and volatility features since we don't have 30 minutes of previous data for the start of the day

In [27]:
df_features = (
    df_features.sort_values("timestamp")
    .reset_index(drop=True)
)

# complete scheduled minute index, including absent observations.
minute_ranges = [
    pd.date_range(
        start=session["market_open"],
        end=session["market_close"],
        freq="min",
        inclusive="left"
    )
    for _, session in schedule.iterrows()
]

expected_index = minute_ranges[0].append(minute_ranges[1:])

# temporary calculation grid; absent bars retain missing prices.
grid = (
    df_features.set_index("timestamp")[["open", "close"]]
    .reindex(expected_index)
)

grid["trading_date"] = grid.index.date

sessions = grid.groupby("trading_date", sort=False)
position = sessions.cumcount()

# exact session opening bar, without skipping missing values.
session_open = sessions["open"].transform(
    lambda x: x.iloc[0]
)

previous_close = sessions["close"].shift(1)

return_reference = previous_close.where(
    position.ne(0),
    grid["open"]
)

returns = np.log(grid["close"] / return_reference)
squared_returns = returns ** 2

# transfer calculated values back to observed bars only.
df_features["log_return_1m"] = returns.reindex(
    df_features["timestamp"]
).to_numpy()

observed_bar = grid[["open", "close"]].notna().all(axis=1)

for window in [5, 15, 30]:

    required_count = np.minimum(position + 1, window)

    # opening price at the exact lookback timestamp.
    lookback_open = sessions["open"].shift(window - 1)

    # shorter opening windows are permitted only near session start.
    lookback_open = lookback_open.where(
        position.ge(window - 1),
        session_open
    )

    observed_count = (
        observed_bar.astype(int)
        .groupby(grid["trading_date"], sort=False)
        .transform(
            lambda x: x.rolling(
                window, min_periods=1
            ).sum()
        )
    )

    momentum = np.log(grid["close"] / lookback_open)
    momentum = momentum.where(
        observed_count.eq(required_count)
    )

    valid_return_count = (
        squared_returns
        .groupby(grid["trading_date"], sort=False)
        .transform(
            lambda x: x.rolling(
                window, min_periods=1
            ).count()
        )
    )

    variance = (
        squared_returns
        .groupby(grid["trading_date"], sort=False)
        .transform(
            lambda x: x.rolling(
                window, min_periods=1
            ).sum()
        )
    )

    volatility = np.sqrt(
        variance.where(valid_return_count.eq(required_count))
    )

    df_features[f"momentum_{window}m"] = momentum.reindex(
        df_features["timestamp"]
    ).to_numpy()

    df_features[f"volatility_{window}m"] = volatility.reindex(
        df_features["timestamp"]
    ).to_numpy()

del grid

print("Multiscale features calculated using exact minute intervals.")

Multiscale features calculated using exact minute intervals.


###Time of day, overnight gap, and session position

In [28]:
# cyclical time encoding
minutes_since_open = df_features["minute_of_day"] - 570
phase = 2 * np.pi * minutes_since_open / 390

df_features["time_sin"] = np.sin(phase)
df_features["time_cos"] = np.cos(phase)

# exact scheduled opening and closing bars
price_lookup = df_features.set_index("timestamp")

daily_prices = pd.DataFrame(
    index=pd.Index(schedule.index.date, name="trading_date")
)

daily_prices["session_open"] = (
    price_lookup["open"]
    .reindex(pd.DatetimeIndex(schedule["market_open"]))
    .to_numpy()
)

closing_bar_times = pd.DatetimeIndex(
    schedule["market_close"] - pd.Timedelta(minutes=1)
)

daily_prices["session_close"] = (
    price_lookup["close"]
    .reindex(closing_bar_times)
    .to_numpy()
)

# shift across scheduled sessions, including any missing sessions.
daily_prices["previous_close"] = (
    daily_prices["session_close"].shift(1)
)

daily_prices["overnight_gap"] = np.log(
    daily_prices["session_open"]
    / daily_prices["previous_close"]
)

df_features["overnight_gap"] = (
    df_features["trading_date"]
    .map(daily_prices["overnight_gap"])
)

# session range through the current bar
sessions = df_features.groupby("trading_date", sort=False)

session_high = sessions["high"].cummax()
session_low = sessions["low"].cummin()
session_range = session_high - session_low

# all scheduled bars from 09:30 through this bar must be present.
valid_range_bar = (
    np.isfinite(
        df_features[["high", "low", "close"]].to_numpy()
    ).all(axis=1)
)

observed_count = (
    pd.Series(
        valid_range_bar.astype(int),
        index=df_features.index
    )
    .groupby(df_features["trading_date"], sort=False)
    .cumsum()
)

complete_so_far = observed_count.eq(
    df_features["minute_index"] + 1
)

range_position = (
    (df_features["close"] - session_low)
    / session_range.replace(0, np.nan)
)

# neutral position only when the observed range is genuinely zero.
range_position = range_position.where(
    session_range.ne(0),
    0.5
)

df_features["session_range_position"] = (
    range_position.where(complete_so_far)
)

print("Session-context features calculated.")

Session-context features calculated.


##Validate all 18 features

In [29]:
# feature structure and dataset integrity
assert len(FEATURE_COLUMNS) == 18
assert len(set(FEATURE_COLUMNS)) == 18

missing_columns = set(FEATURE_COLUMNS) - set(df_features.columns)

assert not missing_columns, (
    f"Missing feature columns: {sorted(missing_columns)}"
)

assert len(df_features) == len(df_prepared), (
    "Feature engineering changed the number of observed rows."
)

assert not df_features.duplicated(
    ["symbol", "timestamp"]
).any(), "Duplicate symbol/timestamps detected."

assert df_features["symbol"].eq("SPY").all()

# select the sample period for validation.
# keep all rows, including warm-up history, in df_features.
sample_period = df_features.loc[
    (df_features["trading_date"] >= SAMPLE_START_DATE)
    & (df_features["trading_date"] < END_DATE.date())
].copy()

assert not sample_period.empty, "No rows in the sample period."

feature_values = sample_period[FEATURE_COLUMNS]

values = feature_values.to_numpy(dtype=np.float64)


# missing and infinite values
feature_quality = pd.DataFrame(
    {
        "missing_rows": np.isnan(values).sum(axis=0),
        "infinite_rows": np.isinf(values).sum(axis=0)
    },
    index=FEATURE_COLUMNS
)

print("Feature availability in the sample period:")
display(
    feature_quality.sort_values(
        "missing_rows",
        ascending=False
    )
)

# Missing values are expected during warm-up and around data gaps.
# Infinite values indicate invalid inputs or calculations.
assert not np.isinf(values).any(), (
    "Infinite feature values detected."
)

# feature bounds
assert (
    sample_period["relative_volume_20d"]
    .dropna()
    .gt(0)
    .all()
), "Relative volume must be positive where available."

for column in [
    "volatility_5m",
    "volatility_15m",
    "volatility_30m"
]:
    assert sample_period[column].dropna().ge(0).all(), (
        f"Negative values detected in {column}."
    )

assert (
    sample_period["session_range_position"]
    .dropna()
    .between(-1e-10, 1 + 1e-10)
    .all()
), "Session range position is outside its expected bounds."

# feature distributions
print("\nFeature distributions in the sample period:")
display(
    feature_values.describe().T[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    ]
)

# overall row availability
complete_rows = np.isfinite(values).all(axis=1)

print("\nFeature integrity checks passed.")
print("Sample-period rows:", f"{len(sample_period):,}")
print("Rows with all 18 features:", f"{complete_rows.sum():,}")
print(
    "Rows with unavailable features:",
    f"{(~complete_rows).sum():,}"
)
print(
    "Sample-period sessions:",
    sample_period["trading_date"].nunique()
)
print("First timestamp:", sample_period["timestamp"].min())
print("Last timestamp:", sample_period["timestamp"].max())


# final input bars for morning predictions
# the completed 09:59 bar becomes available after 10:00 AM Eastern.
morning_cutoffs = sample_period.loc[
    sample_period["timestamp"].dt.hour.eq(9)
    & sample_period["timestamp"].dt.minute.eq(59)
].copy()

cutoff_complete = np.isfinite(
    morning_cutoffs[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
).all(axis=1)

complete_morning_cutoffs = morning_cutoffs.loc[
    cutoff_complete
].copy()

print("\nMorning prediction cutoff checks:")
print("Observed 09:59 bars:", len(morning_cutoffs))
print(
    "09:59 bars with all 18 features:",
    len(complete_morning_cutoffs)
)

print("\nExample complete 09:59 feature vectors:")
display(
    complete_morning_cutoffs[
        ["timestamp", "trading_date"] + FEATURE_COLUMNS
    ].head()
)

print(
    "\nA complete 09:59 feature vector alone does not establish "
    "sample eligibility. The next checks must verify all 30 input "
    "minutes; target construction must also verify future coverage."
)

Feature availability in the sample period:


,missing_rows,infinite_rows
relative_volume_20d,7800,0
session_range_position,2190,0
overnight_gap,780,0
volatility_30m,211,0
momentum_30m,204,0
volatility_15m,106,0
momentum_15m,99,0
volatility_5m,36,0
momentum_5m,29,0
log_return_1m,8,0



Feature distributions in the sample period:


,count,mean,std,min,25%,50%,75%,max
log_return_1m,1041308.0,5.150184e-07,0.000428,-0.014898,-0.000158,0.000000e+00,0.000162,0.025418
candle_range,1041316.0,4.971711e-04,0.000485,0.000000,0.000224,3.659071e-04,0.000606,0.032776
candle_body,1041316.0,3.616906e-07,0.000427,-0.014768,-0.000157,0.000000e+00,0.000161,0.026093
upper_wick,1041316.0,1.182011e-04,0.000175,0.000000,0.000026,7.032349e-05,0.000149,0.029739
lower_wick,1041316.0,1.202962e-04,0.000178,0.000000,0.000027,7.174631e-05,0.000152,0.027584
log_volume,1041316.0,1.166320e+01,0.872641,7.365180,11.067603,1.161584e+01,12.210180,16.990827
relative_volume_20d,1033516.0,1.057680e+00,1.318644,0.010994,0.490795,7.546124e-01,1.199763,172.665566
momentum_5m,1041287.0,2.252598e-06,0.000949,-0.034191,-0.000348,1.499089e-05,0.000366,0.044727
momentum_15m,1041217.0,7.245108e-06,0.001602,-0.039384,-0.000595,4.125687e-05,0.000647,0.059221
momentum_30m,1041112.0,1.602738e-05,0.002213,-0.047115,-0.000832,7.530688e-05,0.000928,0.070343



Feature integrity checks passed.
Sample-period rows: 1,041,316
Rows with all 18 features: 1,030,936
Rows with unavailable features: 10,380
Sample-period sessions: 2680
First timestamp: 2016-01-04 09:30:00-05:00
Last timestamp: 2026-08-31 15:59:00-04:00

Morning prediction cutoff checks:
Observed 09:59 bars: 2680
09:59 bars with all 18 features: 2655

Example complete 09:59 feature vectors:


,timestamp,trading_date,log_return_1m,candle_range,candle_body,upper_wick,lower_wick,log_volume,relative_volume_20d,momentum_5m,momentum_15m,momentum_30m,volatility_5m,volatility_15m,volatility_30m,time_sin,time_cos,overnight_gap,session_range_position,vwap_deviation
7829,2016-02-02 09:59:00-05:00,2016-02-02,-0.000105,0.000366,-0.000052,0.000105,0.000209,12.292961,0.510671,0.000994,-0.000157,-0.004019,0.001330,0.002277,0.003286,0.450398,0.892828,-0.008972,0.290909,0.000034
8210,2016-02-03 09:59:00-05:00,2016-02-03,-0.001630,0.001628,-0.001576,0.000000,0.000053,13.112180,1.149241,-0.000473,-0.002784,-0.006973,0.002054,0.003739,0.004118,0.450398,0.892828,0.006710,0.066667,-0.000661
8600,2016-02-04 09:59:00-05:00,2016-02-04,0.000576,0.001467,0.000576,0.000786,0.000105,12.973501,0.969836,-0.000105,0.004302,0.001624,0.001413,0.003055,0.006344,0.450398,0.892828,-0.002514,0.751773,-0.000262
8990,2016-02-05 09:59:00-05:00,2016-02-05,-0.000684,0.000894,-0.000683,0.000000,0.000210,12.731153,0.746435,0.000053,-0.001682,-0.004776,0.001682,0.003148,0.004197,0.450398,0.892828,-0.002928,0.275362,-0.000145
9380,2016-02-08 09:59:00-05:00,2016-02-08,-0.000975,0.001191,-0.000947,0.000135,0.000108,13.596683,1.767753,0.000379,-0.005080,-0.006426,0.001942,0.004112,0.005249,0.450398,0.892828,-0.011667,0.193750,-0.000497



A complete 09:59 feature vector alone does not establish sample eligibility. The next checks must verify all 30 input minutes; target construction must also verify future coverage.


##verify all 30 minutes of the morning input

In [30]:
# select the 9:30–9:59 AM input window
morning_df = df_features[
    df_features["timestamp"].dt.hour.eq(9)
    & df_features["timestamp"].dt.minute.between(30, 59)
].copy()

# check all 18 features for missing or infinite values
morning_df["features_complete"] = np.isfinite(
    morning_df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)
).all(axis=1)

# summarize completeness for each session
morning_quality = morning_df.groupby("trading_date").agg(
    bars=("timestamp", "size"),
    complete_feature_rows=("features_complete", "sum")
)

print("Morning window completeness:")
display(morning_quality.describe())

print("\nSessions without 30 complete feature rows:")
display(
    morning_quality[
        morning_quality["bars"].ne(30)
        | morning_quality["complete_feature_rows"].ne(30)
    ]
)

Morning window completeness:


,bars,complete_feature_rows
count,2680.000000,2680.000000
mean,29.982836,29.732836
std,0.474513,2.787397
min,16.000000,0.000000
25%,30.000000,30.000000
50%,30.000000,30.000000
75%,30.000000,30.000000
max,30.000000,30.000000



Sessions without 30 complete feature rows:


,bars,complete_feature_rows
trading_date,,
2016-01-04,30,0
2016-01-05,30,0
2016-01-06,30,0
2016-01-07,30,0
2016-01-08,30,0
2016-01-11,30,0
2016-01-12,30,0
2016-01-13,30,0
2016-01-14,30,0


##Validating rolling momentum and volatility

In [31]:
multiscale_features = [
    "momentum_5m",
    "momentum_15m",
    "momentum_30m",
    "volatility_5m",
    "volatility_15m",
    "volatility_30m"
]

print("Missing multiscale feature values:")
display(
    df_features[multiscale_features]
    .isna()
    .sum()
)

print("\nInfinite values:")
print(
    np.isinf(
        df_features[multiscale_features].to_numpy()
    ).sum()
)

assert not np.isinf(
    df_features[multiscale_features].to_numpy(dtype=np.float64)
).any()

print(
    "\nMissing values are retained for gap-affected bars. "
    "Affected model samples must be excluded."
)

Missing multiscale feature values:


,0
momentum_5m,29
momentum_15m,99
momentum_30m,204
volatility_5m,36
volatility_15m,106
volatility_30m,211



Infinite values:
0

Missing values are retained for gap-affected bars. Affected model samples must be excluded.


In [32]:
# choose a session with a full 20-day volume history
sample_date = pd.Timestamp("2024-02-01").date()

morning_sample = df_features[
    (df_features["trading_date"] == sample_date) &
    (df_features["timestamp"].dt.hour == 9) &
    (df_features["timestamp"].dt.minute.between(30, 59))
].copy()

display(
    morning_sample[
        [
            "timestamp",
            "log_return_1m",
            "momentum_5m",
            "momentum_15m",
            "momentum_30m",
            "volatility_5m",
            "volatility_15m",
            "volatility_30m"
        ]
    ]
)

,timestamp,log_return_1m,momentum_5m,momentum_15m,momentum_30m,volatility_5m,volatility_15m,volatility_30m
790066,2024-02-01 09:30:00-05:00,0.001371,0.001371,0.001371,0.001371,0.001371,0.001371,0.001371
790067,2024-02-01 09:31:00-05:00,-0.000010,0.001361,0.001361,0.001361,0.001371,0.001371,0.001371
790068,2024-02-01 09:32:00-05:00,-0.000443,0.000918,0.000918,0.000918,0.001441,0.001441,0.001441
790069,2024-02-01 09:33:00-05:00,0.000464,0.001382,0.001382,0.001382,0.001514,0.001514,0.001514
790070,2024-02-01 09:34:00-05:00,0.000479,0.001860,0.001860,0.001860,0.001588,0.001588,0.001588
790071,2024-02-01 09:35:00-05:00,-0.000479,0.000010,0.001382,0.001382,0.000933,0.001658,0.001658
790072,2024-02-01 09:36:00-05:00,0.000103,0.000144,0.001485,0.001485,0.000938,0.001661,0.001661
790073,2024-02-01 09:37:00-05:00,0.000031,0.000587,0.001515,0.001515,0.000827,0.001662,0.001662
790074,2024-02-01 09:38:00-05:00,-0.000402,-0.000247,0.001114,0.001114,0.000794,0.001710,0.001710
790075,2024-02-01 09:39:00-05:00,0.000268,-0.000495,0.001381,0.001381,0.000688,0.001730,0.001730


##Validate all morning Transformer input sequences

In [33]:
# select the morning input window
morning_df = df_features[
    (df_features["timestamp"].dt.hour == 9) &
    (df_features["timestamp"].dt.minute.between(30, 59))
].copy()

# count observations for each trading session
morning_counts = (
    morning_df.groupby("trading_date")
    .size()
)

# check whether each minute has all 18 features
morning_df["features_complete"] = (
    np.isfinite(
        morning_df[FEATURE_COLUMNS].to_numpy(
            dtype=np.float64
        )
    ).all(axis=1)
)

# summarize by session
morning_quality = (
    morning_df.groupby("trading_date")
    .agg(
        bars=("timestamp", "size"),
        complete_rows=("features_complete", "sum")
    )
)

# eligible sessions
eligible_sessions = morning_quality[
    (morning_quality["bars"] == 30) &
    (morning_quality["complete_rows"] == 30)
].index

print("Total trading sessions:", len(morning_quality))
print("Complete morning sequences:", len(eligible_sessions))
print("Incomplete morning sequences:",
      len(morning_quality) - len(eligible_sessions))

print("\nMorning sequence validation:")
display(morning_quality.describe())

print("\nIncomplete sessions:")
display(
    morning_quality.loc[
        ~morning_quality.index.isin(eligible_sessions)
    ]
)

Total trading sessions: 2680
Complete morning sequences: 2655
Incomplete morning sequences: 25

Morning sequence validation:


,bars,complete_rows
count,2680.000000,2680.000000
mean,29.982836,29.732836
std,0.474513,2.787397
min,16.000000,0.000000
25%,30.000000,30.000000
50%,30.000000,30.000000
75%,30.000000,30.000000
max,30.000000,30.000000



Incomplete sessions:


,bars,complete_rows
trading_date,,
2016-01-04,30,0
2016-01-05,30,0
2016-01-06,30,0
2016-01-07,30,0
2016-01-08,30,0
2016-01-11,30,0
2016-01-12,30,0
2016-01-13,30,0
2016-01-14,30,0


##Build preliminary input tensor

In [34]:
# keep only eligible sessions
model_input_df = morning_df[
    morning_df["trading_date"].isin(eligible_sessions)
].copy()

model_input_df = model_input_df.sort_values(
    ["trading_date", "timestamp"]
)

# build input tensor
X = np.stack([
    group[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    for _, group in model_input_df.groupby(
        "trading_date", sort=True
    )
])

# corresponding session dates
sample_dates = np.array(
    sorted(eligible_sessions)
)

assert X.shape == (
    len(eligible_sessions),
    30,
    18
)

assert np.isfinite(X).all()

print("Input tensor shape:", X.shape)
print("Number of samples:", len(sample_dates))
print("First sample date:", sample_dates[0])
print("Last sample date:", sample_dates[-1])

print("\nExample input tensor:")
print(X[0])

Input tensor shape: (2655, 30, 18)
Number of samples: 2655
First sample date: 2016-02-02
Last sample date: 2026-08-31

Example input tensor:
[[-8.85993533e-04  9.89789492e-04 -8.85601155e-04  5.20941867e-05
   5.20941867e-05  1.45042658e+01  1.00783050e+00 -8.85993533e-04
  -8.85993533e-04 -8.85993533e-04  8.85993533e-04  8.85993533e-04
   8.85993533e-04  0.00000000e+00  1.00000000e+00 -8.97192582e-03
   5.26315793e-02 -4.99453221e-04]
 [-1.04286162e-04  7.03913218e-04 -7.82125790e-05  7.82125790e-05
   5.47488045e-04  1.35968304e+01  9.13193285e-01 -9.90279717e-04
  -9.90279717e-04 -9.90279717e-04  8.92109936e-04  8.92109936e-04
   8.92109936e-04  1.61100347e-02  9.99870241e-01 -8.97192582e-03
   3.44262302e-01  2.44602765e-04]
 [-6.25945453e-04  8.34289298e-04 -6.77860051e-04  1.04286162e-04
   5.21430811e-05  1.32328453e+01  6.57414794e-01 -1.61622511e-03
  -1.61622511e-03 -1.61622511e-03  1.08980178e-03  1.08980178e-03
   1.08980178e-03  3.22158895e-02  9.99480963e-01 -8.97192582e-

## Build X, Y and metadata

The model takes X_raw as its input—after we normalize it.


During training, it also needs y_raw, the three actual future returns, to measure prediction errors and learn.


Metadata stays outside the model. We use it to assign chronological splits, check alignment, and connect predictions back to dates and prices for evaluation.

In [35]:
# build inputs, targets, and sample metadata
HORIZONS = [15, 30, 60]
TARGET_COLUMNS = [
    "return_15m_pct",
    "return_30m_pct",
    "return_60m_pct"
]

assert not df_features.duplicated(
    ["symbol", "timestamp"]
).any()

assert df_features["symbol"].eq("SPY").all()

features_by_date = {
    day: group.set_index("timestamp").sort_index()
    for day, group in df_features.groupby(
        "trading_date", sort=True
    )
}

input_arrays = []
target_arrays = []
metadata_rows = []
audit_rows = []

for session_date, session in tqdm(
    schedule.iterrows(),
    total=len(schedule),
    desc="Building model samples"
):
    day = session_date.date()

    if day < SAMPLE_START_DATE or day >= END_DATE.date():
        continue

    # chronological partitions specified in the proposal
    if day < date(2025, 1, 1):
        partition = "train"
    elif day < date(2026, 1, 1):
        partition = "validation"
    else:
        partition = "test"

    expected_times = pd.date_range(
        start=session["market_open"],
        end=session["market_close"],
        freq="min",
        inclusive="left"
    )

    source = features_by_date.get(day)

    if source is None:
        audit_rows.append({
            "trading_date": day,
            "partition": partition,
            "input_start": session["market_open"],
            "status": "missing_session"
        })
        continue

    # preserve missing observations as NaNs.
    session_grid = source.reindex(expected_times)

    feature_array = session_grid[
        FEATURE_COLUMNS
    ].to_numpy(dtype=np.float64)

    close_array = session_grid["close"].to_numpy(
        dtype=np.float64
    )

    # 30 input bars plus 60 future bars must fit in the session.
    if partition == "train":
        start_positions = range(
            0, len(expected_times) - 90 + 1, 30
        )
    else:
        start_positions = [0]

    for start in start_positions:
        input_end = start + 29
        final_target = input_end + max(HORIZONS)

        audit_record = {
            "trading_date": day,
            "partition": partition,
            "input_start": expected_times[start]
        }

        if final_target >= len(expected_times):
            audit_record["status"] = "outside_session"
            audit_rows.append(audit_record)
            continue

        input_values = feature_array[start:input_end + 1]

        if not np.isfinite(input_values).all():
            audit_record["status"] = "incomplete_input_features"
            audit_rows.append(audit_record)
            continue

        # require every close from the final input bar through
        # the longest target horizon, including intermediate bars.
        required_closes = close_array[
            input_end:final_target + 1
        ]

        if (
            not np.isfinite(required_closes).all()
            or not (required_closes > 0).all()
        ):
            audit_record["status"] = "incomplete_future_prices"
            audit_rows.append(audit_record)
            continue

        reference_close = close_array[input_end]

        future_closes = np.array([
            close_array[input_end + horizon]
            for horizon in HORIZONS
        ])

        returns_pct = 100 * (
            future_closes / reference_close - 1
        )

        input_arrays.append(input_values.astype(np.float32))
        target_arrays.append(returns_pct)

        metadata = {
            "trading_date": day,
            "partition": partition,
            "input_start": expected_times[start],
            "last_input_bar": expected_times[input_end],
            # earliest theoretical availability; receipt is later.
            "input_bar_end": (
                expected_times[input_end]
                + pd.Timedelta(minutes=1)
            ),
            "reference_close": reference_close
        }

        for horizon, future_close, target in zip(
            HORIZONS, future_closes, returns_pct
        ):
            target_bar = expected_times[input_end + horizon]

            metadata[f"target_bar_{horizon}m"] = target_bar
            metadata[f"target_bar_end_{horizon}m"] = (
                target_bar + pd.Timedelta(minutes=1)
            )
            metadata[f"future_close_{horizon}m"] = future_close
            metadata[f"return_{horizon}m_pct"] = target

        metadata_rows.append(metadata)

        audit_record["status"] = "accepted"
        audit_rows.append(audit_record)

assert input_arrays, "No eligible samples were constructed."

# unnormalized datasets, in matching sample order
X_raw = np.stack(input_arrays)
y_raw = np.stack(target_arrays).astype(np.float64)

sample_metadata = pd.DataFrame(metadata_rows)
sample_audit = pd.DataFrame(audit_rows)

# alignment and numerical checks
assert X_raw.shape == (
    len(sample_metadata), 30, len(FEATURE_COLUMNS)
)

assert y_raw.shape == (len(sample_metadata), 3)

assert np.isfinite(X_raw).all()
assert np.isfinite(y_raw).all()

assert not sample_metadata.duplicated(
    ["trading_date", "input_start"]
).any()

assert (
    sample_metadata.groupby("trading_date")["partition"]
    .nunique()
    .eq(1)
    .all()
)

np.testing.assert_allclose(
    y_raw,
    sample_metadata[TARGET_COLUMNS].to_numpy()
)

print("Input shape:", X_raw.shape)
print("Target shape:", y_raw.shape)
print("Target order:", TARGET_COLUMNS)

print("\nAccepted samples by partition:")
display(
    sample_metadata.groupby("partition").agg(
        samples=("trading_date", "size"),
        sessions=("trading_date", "nunique"),
        first_date=("trading_date", "min"),
        last_date=("trading_date", "max")
    )
)

print("\nSample acceptance/exclusion summary:")
display(
    pd.crosstab(
        sample_audit["partition"],
        sample_audit["status"]
    )
)

print("\nExample targets and reference prices:")
display(sample_metadata.head())

Building model samples:   0%|          | 0/2680 [00:00<?, ?it/s]

Input shape: (24908, 30, 18)
Target shape: (24908, 3)
Target order: ['return_15m_pct', 'return_30m_pct', 'return_60m_pct']

Accepted samples by partition:


,samples,sessions,first_date,last_date
partition,,,,
test,166,166,2026-01-02,2026-08-31
train,24492,2239,2016-02-02,2024-12-31
validation,250,250,2025-01-02,2025-12-31



Sample acceptance/exclusion summary:


status,accepted,incomplete_future_prices,incomplete_input_features
partition,,,
test,166,0,0
train,24492,8,296
validation,250,0,0



Example targets and reference prices:


,trading_date,partition,input_start,last_input_bar,input_bar_end,reference_close,target_bar_15m,target_bar_end_15m,future_close_15m,return_15m_pct,target_bar_30m,target_bar_end_30m,future_close_30m,return_30m_pct,target_bar_60m,target_bar_end_60m,future_close_60m,return_60m_pct
0,2016-02-02,train,2016-02-02 09:30:00-05:00,2016-02-02 09:59:00-05:00,2016-02-02 10:00:00-05:00,191.19,2016-02-02 10:14:00-05:00,2016-02-02 10:15:00-05:00,191.1350,-0.028767,2016-02-02 10:29:00-05:00,2016-02-02 10:30:00-05:00,191.10,-0.047074,2016-02-02 10:59:00-05:00,2016-02-02 11:00:00-05:00,191.00,-0.099378
1,2016-02-03,train,2016-02-03 09:30:00-05:00,2016-02-03 09:59:00-05:00,2016-02-03 10:00:00-05:00,190.08,2016-02-03 10:14:00-05:00,2016-02-03 10:15:00-05:00,188.2499,-0.962805,2016-02-03 10:29:00-05:00,2016-02-03 10:30:00-05:00,187.83,-1.183712,2016-02-03 10:59:00-05:00,2016-02-03 11:00:00-05:00,188.93,-0.605008
2,2016-02-03,train,2016-02-03 10:00:00-05:00,2016-02-03 10:29:00-05:00,2016-02-03 10:30:00-05:00,187.83,2016-02-03 10:44:00-05:00,2016-02-03 10:45:00-05:00,187.9700,0.074535,2016-02-03 10:59:00-05:00,2016-02-03 11:00:00-05:00,188.93,0.585636,2016-02-03 11:29:00-05:00,2016-02-03 11:30:00-05:00,189.15,0.702763
3,2016-02-03,train,2016-02-03 10:30:00-05:00,2016-02-03 10:59:00-05:00,2016-02-03 11:00:00-05:00,188.93,2016-02-03 11:14:00-05:00,2016-02-03 11:15:00-05:00,188.3800,-0.291113,2016-02-03 11:29:00-05:00,2016-02-03 11:30:00-05:00,189.15,0.116445,2016-02-03 11:59:00-05:00,2016-02-03 12:00:00-05:00,189.26,0.174668
4,2016-02-03,train,2016-02-03 11:00:00-05:00,2016-02-03 11:29:00-05:00,2016-02-03 11:30:00-05:00,189.15,2016-02-03 11:44:00-05:00,2016-02-03 11:45:00-05:00,189.6101,0.243246,2016-02-03 11:59:00-05:00,2016-02-03 12:00:00-05:00,189.26,0.058155,2016-02-03 12:29:00-05:00,2016-02-03 12:30:00-05:00,189.18,0.015860


##Separate into training, validation and test datasets. Normalize using training statistics.

In [36]:
# check sample alignment
assert len(X_raw) == len(y_raw) == len(sample_metadata)
assert X_raw.shape[1:] == (30, len(FEATURE_COLUMNS))
assert y_raw.shape[1:] == (len(TARGET_COLUMNS),)

assert np.isfinite(X_raw).all()
assert np.isfinite(y_raw).all()

sample_metadata = sample_metadata.reset_index(drop=True)

np.testing.assert_allclose(
    y_raw,
    sample_metadata[TARGET_COLUMNS].to_numpy()
)

# verify chronological partition assignments
sample_days = pd.to_datetime(
    sample_metadata["trading_date"]
)

expected_partition = np.where(
    sample_days < pd.Timestamp("2025-01-01"),
    "train",
    np.where(
        sample_days < pd.Timestamp("2026-01-01"),
        "validation",
        "test"
    )
)

assert (
    sample_metadata["partition"].to_numpy()
    == expected_partition
).all(), "Partition assignments do not match proposal dates."

assert sample_days.ge(pd.Timestamp("2016-01-01")).all()
assert sample_days.lt(pd.Timestamp("2026-09-01")).all()

partition_indices = {
    partition: np.flatnonzero(
        sample_metadata["partition"].eq(partition).to_numpy()
    )
    for partition in ["train", "validation", "test"]
}

for partition, indices in partition_indices.items():
    assert len(indices) > 0, f"No samples in {partition}."

assert (
    sample_days.iloc[partition_indices["train"]].max()
    < sample_days.iloc[partition_indices["validation"]].min()
)

assert (
    sample_days.iloc[partition_indices["validation"]].max()
    < sample_days.iloc[partition_indices["test"]].min()
)

# fit normalization using training samples only
train_indices = partition_indices["train"]

# one mean and standard deviation per input feature.
# pool the training samples and their 30 time positions.
training_inputs = X_raw[train_indices]

feature_mean = training_inputs.mean(
    axis=(0, 1), dtype=np.float64
)

feature_std = training_inputs.std(
    axis=(0, 1), dtype=np.float64
)

# a constant feature gets scale 1, avoiding division by zero.
feature_scale = np.where(
    feature_std == 0, 1.0, feature_std
)

# one mean and standard deviation per target horizon.
training_targets = y_raw[train_indices]

target_mean = training_targets.mean(
    axis=0, dtype=np.float64
)

target_std = training_targets.std(
    axis=0, dtype=np.float64
)

target_scale = np.where(
    target_std == 0, 1.0, target_std
)

del training_inputs, training_targets

# retain everything needed to repeat or reverse normalization.
normalization = {
    "feature_columns": list(FEATURE_COLUMNS),
    "target_columns": list(TARGET_COLUMNS),
    "feature_mean": feature_mean,
    "feature_scale": feature_scale,
    "target_mean": target_mean,
    "target_scale": target_scale
}

# build the three normalized datasets
datasets = {}

for partition, indices in partition_indices.items():

    X_partition = (
        (X_raw[indices] - feature_mean)
        / feature_scale
    ).astype(np.float32)

    y_partition = (
        (y_raw[indices] - target_mean)
        / target_scale
    ).astype(np.float32)

    assert np.isfinite(X_partition).all()
    assert np.isfinite(y_partition).all()

    datasets[partition] = {
        "X": X_partition,
        "y": y_partition,
        "y_return_pct": y_raw[indices].copy(),
        "metadata": (
            sample_metadata.iloc[indices]
            .reset_index(drop=True)
            .copy()
        )
    }

# convenient names for later model code
X_train = datasets["train"]["X"]
y_train = datasets["train"]["y"]

X_val = datasets["validation"]["X"]
y_val = datasets["validation"]["y"]

X_test = datasets["test"]["X"]
y_test = datasets["test"]["y"]

# verify target normalization can be reversed
for partition, data in datasets.items():

    recovered_returns = (
        data["y"].astype(np.float64) * target_scale
        + target_mean
    )

    np.testing.assert_allclose(
        recovered_returns,
        data["y_return_pct"],
        rtol=1e-5,
        atol=1e-6
    )

# summarize the resulting datasets
summary_rows = []

for partition, data in datasets.items():
    metadata = data["metadata"]

    summary_rows.append({
        "partition": partition,
        "samples": len(metadata),
        "sessions": metadata["trading_date"].nunique(),
        "first_date": metadata["trading_date"].min(),
        "last_date": metadata["trading_date"].max(),
        "X_shape": data["X"].shape,
        "y_shape": data["y"].shape
    })

print("Dataset partitions:")
display(pd.DataFrame(summary_rows))

print("\nFeature normalization fitted on training samples:")
display(
    pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "training_mean": feature_mean,
        "training_std": feature_std,
        "applied_scale": feature_scale
    })
)

print("\nTarget normalization fitted on training samples:")
display(
    pd.DataFrame({
        "target": TARGET_COLUMNS,
        "training_mean_pct": target_mean,
        "training_std_pct": target_std,
        "applied_scale": target_scale
    })
)

print("\nPartition and normalization checks passed.")

Dataset partitions:


,partition,samples,sessions,first_date,last_date,X_shape,y_shape
0,train,24492,2239,2016-02-02,2024-12-31,"(24492, 30, 18)","(24492, 3)"
1,validation,250,250,2025-01-02,2025-12-31,"(250, 30, 18)","(250, 3)"
2,test,166,166,2026-01-02,2026-08-31,"(166, 30, 18)","(166, 3)"



Feature normalization fitted on training samples:


,feature,training_mean,training_std,applied_scale
0,log_return_1m,7.496100e-07,0.000407,0.000407
1,candle_range,4.826295e-04,0.000452,0.000452
2,candle_body,4.960686e-07,0.000406,0.000406
3,upper_wick,1.150778e-04,0.000168,0.000168
4,lower_wick,1.161149e-04,0.000163,0.000163
5,log_volume,1.156894e+01,0.840308,0.840308
6,relative_volume_20d,1.060344e+00,1.396819,1.396819
7,momentum_5m,3.377987e-06,0.000904,0.000904
8,momentum_15m,9.588846e-06,0.001527,0.001527
9,momentum_30m,1.866467e-05,0.002126,0.002126



Target normalization fitted on training samples:


,target,training_mean_pct,training_std_pct,applied_scale
0,return_15m_pct,-0.000006,0.150588,0.150588
1,return_30m_pct,0.002042,0.205623,0.205623
2,return_60m_pct,0.003543,0.296170,0.296170



Partition and normalization checks passed.


##Save the data to google drive

In [37]:
# save the prepared datasets to Google Drive
import json

bundle_timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

DATASET_DIR = (
    PROJECT_DIR
    / "datasets"
    / f"spy_quantiles_{bundle_timestamp}"
)

DATASET_DIR.mkdir(parents=True, exist_ok=False)

# save arrays and matching metadata
for partition, data in datasets.items():

    indices = partition_indices[partition]

    np.savez_compressed(
        DATASET_DIR / f"{partition}.npz",
        X=data["X"],
        y=data["y"],
        X_raw=X_raw[indices],
        y_return_pct=data["y_return_pct"]
    )

    data["metadata"].to_parquet(
        DATASET_DIR / f"{partition}_metadata.parquet",
        index=False
    )


# save training-fitted normalization parameters

np.savez_compressed(
    DATASET_DIR / "normalization.npz",
    feature_columns=np.asarray(FEATURE_COLUMNS, dtype=str),
    target_columns=np.asarray(TARGET_COLUMNS, dtype=str),
    feature_mean=feature_mean,
    feature_scale=feature_scale,
    target_mean=target_mean,
    target_scale=target_scale
)

# save source observations, features, and exclusions

df_clean.to_parquet(
    DATASET_DIR / "cleaned_bars.parquet",
    index=False
)

df_features.to_parquet(
    DATASET_DIR / "engineered_features.parquet",
    index=False
)

sample_audit.to_parquet(
    DATASET_DIR / "sample_audit.parquet",
    index=False
)

# save dataset definitions

manifest = {
    "created_utc": bundle_timestamp,
    "symbol": "SPY",
    "feed": "SIP",
    "price_adjustment": "raw",
    "timezone": "America/New_York",
    "input_minutes": 30,
    "feature_columns": list(FEATURE_COLUMNS),
    "target_columns": list(TARGET_COLUMNS),
    "horizons_minutes": list(HORIZONS),
    "prediction_quantiles": [0.1, 0.5, 0.9],
    "target_formula": (
        "100 * (future_close / last_input_close - 1)"
    ),
    "target_units": "percentage points",
    "training_sampling": (
        "30-minute inputs starting every 30 minutes "
        "from session open"
    ),
    "validation_test_sampling": "09:30–09:59 inputs only",
    "normalization_fit": "accepted training samples only",
    "feature_normalization": (
        "one mean and population standard deviation per "
        "feature, pooled across training samples and time positions"
    ),
    "target_normalization": (
        "one mean and population standard deviation per horizon"
    ),
    "gap_policy": (
        "No price filling. Require finite input features and "
        "all closing-price observations through the 60-minute target."
    ),
    "availability_note": (
        "Bar timestamps identify interval starts. Prediction "
        "requires receipt of the completed final input bar."
    ),
    "partitions": {
        partition: {
            "samples": len(data["metadata"]),
            "sessions": int(
                data["metadata"]["trading_date"].nunique()
            ),
            "first_date": str(
                data["metadata"]["trading_date"].min()
            ),
            "last_date": str(
                data["metadata"]["trading_date"].max()
            )
        }
        for partition, data in datasets.items()
    }
}

with open(
    DATASET_DIR / "manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(manifest, file, indent=2)

# reload and verify the saved partition files
for partition, data in datasets.items():

    saved_metadata = pd.read_parquet(
        DATASET_DIR / f"{partition}_metadata.parquet"
    )

    with np.load(
        DATASET_DIR / f"{partition}.npz",
        allow_pickle=False
    ) as saved:

        np.testing.assert_array_equal(
            saved["X"], data["X"]
        )

        np.testing.assert_array_equal(
            saved["y"], data["y"]
        )

        np.testing.assert_array_equal(
            saved["X_raw"],
            X_raw[partition_indices[partition]]
        )

        np.testing.assert_array_equal(
            saved["y_return_pct"],
            saved_metadata[TARGET_COLUMNS].to_numpy()
        )

        assert len(saved_metadata) == len(saved["X"])

    # parquet may represent Python dates differently on reload.
    pd.testing.assert_frame_equal(
        saved_metadata.assign(
            trading_date=pd.to_datetime(
                saved_metadata["trading_date"]
            )
        ),
        data["metadata"].assign(
            trading_date=pd.to_datetime(
                data["metadata"]["trading_date"]
            )
        )
    )

print("Datasets saved and partition reload checks passed.")
print("Dataset folder:", DATASET_DIR)

print("\nSaved files:")
for path in sorted(DATASET_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1024**2:.2f} MB")

Datasets saved and partition reload checks passed.
Dataset folder: /content/drive/MyDrive/SPY_DL_Project/datasets/spy_quantiles_20261104T210534_492631Z

Saved files:
cleaned_bars.parquet: 34.12 MB
engineered_features.parquet: 151.99 MB
manifest.json: 0.00 MB
normalization.npz: 0.00 MB
sample_audit.parquet: 0.26 MB
test.npz: 0.55 MB
test_metadata.parquet: 0.03 MB
train.npz: 81.16 MB
train_metadata.parquet: 3.30 MB
validation.npz: 0.83 MB
validation_metadata.parquet: 0.04 MB
